# Patch-Wise Underwater Image Quality Assessment
## Multi-Scale Transformer + Degradation-Aware Hybrid Architecture

---

**Architecture source:** *Three Novel Deep Learning Solutions for Patch-Wise Underwater Image Quality Assessment* (uploaded design document).

**Implementation:** Research prototype. Results require experimental validation on a real UIQA dataset with human MOS labels.

---


# v3 changes (from v2)

Each item maps to the review list.

| Item | Where | Change |
|---|---|---|
| 1. Protocol | CFG, this cell | **Not matched** (see status below). |
| 2. Variance | CFG `KFOLD=5`, `SEEDS=[42,43,44]`; section 21d | Each seed trains a 5-fold ensemble. Results are reported as mean ± std over seeds. The data split and fold partition are fixed; only training seeds vary. |
| 3. Features | CFG `CNN_BACKBONE`, `CNN_INPUT_SIZE`; CNN encoder (section 12); ablations E/F | DINOv2 backbone option via timm. Patches are resized to 224 for the ViT. `IMAGE_SIZE=384` is ablation D. KADID checkpoints are keyed by backbone and size. |
| 4. Degradation | Section 15b; `USE_FILM`, `USE_DEG_PRETRAIN` | Stage 1: a degradation estimator is pretrained on synthetic haze, colour-cast and blur with known levels. It uses pool images only and no MOS. Stage 2: its embedding FiLM-conditions the Transformer tokens (zero-initialised, so the model starts as v2). |
| 5. Loss / labels | `MOS_LOSS`, `HUBER_DELTA`; UIQALoss (section 18); section 21e | Huber MOS loss is the default; MSE is still available (ablation B). The label audit lists the worst out-of-fold images and shows the four named ones. |
| 6. Calibration | Section 21b cell; section 21d | 5-parameter logistic map fitted on out-of-fold validation predictions. It is applied to each fold model's test output and then averaged. v2 fitted on training predictions, which are in-sample. Restarts are chosen by fit MSE, not by correlation on the same data. |

**Status**

- **Protocol (item 1) is not matched.** The EDANet repo README (github.com/yia-yuese/EDANet) gives no split ratio, resize or crop. The UWIQA source paper (ScienceDirect S0923596521000503) or the authors' data loader would need to be checked. Until then, do not claim a comparison with EDANet.
- **The degradation design differs from EDANet.** EDANet's DIDNet is pretrained to predict a residual against a pseudo-reference image. UWIQA has no clean references, so this notebook uses synthetic levels instead. That is a different pretext task and is untested.
- **Not executed.** The dataset zip was not in the upload, and the sandbox could not run PyTorch. Only syntax and the numpy-only logistic and metric code were checked. Run the notebook top to bottom before reading any number.
- **Compute.** One seed means 5 full fold trainings. The default 3 seeds means 15, and each ablation repeats that. Start with `SEEDS = [42]` to check the pipeline.
- **Explainability (section 25) and per-image saving (section 27)** still use the single-run `model`. With `KFOLD >= 2` they show untrained weights until pointed at a fold checkpoint.
- **Partial fine-tuning** is all-or-nothing in this version. The whole ViT is unfrozen at 0.2 x LR after `FREEZE_EPOCHS`. A frozen light-head run is ablation F.
- **MOS scale.** `HUBER_DELTA = 0.1` and the MOS bins in the interval analysis assume MOS in [0, 1]. Confirm this against the spreadsheet.


**v3b:** class-balanced training loader with label-preserving augmentation, plus a before/after class-distribution bar graph (section 8b). Controlled by `BALANCE_CLASSES`, `CLASS_BIN_WIDTH`, `BALANCE_POWER`, `BALANCE_ROT90`. Not executed here (no dataset or PyTorch in the sandbox).

**v3c:** degradation estimator uses GroupNorm (`DEG_NORM`); Transformer keeps LayerNorm; pretrained CNN keeps its BatchNorm. One *iteration* = one full run that trains until early-stopping patience (30) is exhausted and then calculates metrics; `KFOLD = 10` gives 10 iterations with one seed. Per-epoch printing, per-epoch CSV (`outputs/logs/<tag>_epochs.csv`), `_last.pt` / `_best.pt` checkpoints and a per-iteration table (`outputs/logs/iterations_summary.csv`).

**v4a (protocol fixes):** train/val/test split and the k folds are stratified by MOS bin (`strat_labels`, `make_folds`); the degradation estimator is pretrained inside each fold on that fold's training images only, and cached checkpoints are reused only when the image-set signature matches.

**v4b:** split changed to 70/10/20 (train/val/test). With `KFOLD >= 2` the train+val images (80%) form the pool that is split into folds; the 20% test set stays untouched.

**v4c:** adds a MOS-direction check and a near-duplicate check (section 8c), a degradation / complexity ablation on one fold (21d-4) and a test-set error audit (21e-2).

**v4c+:** `USE_KADID_INIT` switch and a KADID check (21d-5) comparing ImageNet-only against ImageNet-then-KADID on one fold.

**v4d:** pretrained Swin Transformer backbones (`CNN_BACKBONE = "swin_t"` or `"swin_s"`, timm weights) used as whole-image encoders whose feature map is pooled to the same token grids as the patch path; backbone check 21d-6 and Swin entries in the 21f ablation list.


## 0 · Dataset Setup — Unzip UIQA Dataset

Place **`UWIQA Dataset.zip`** in the same folder as this notebook, then run the cell below.

Expected zip structure (unzipped automatically):
```
UWIQA Dataset.zip
└── UWIQA/                 ← images named 0001.png, 0002.png, 0003.png ...
└── IQA Value.xlsx
```

After running this cell:
- Images will be accessible at `./UWIQA/`
- Quality labels will be accessible at `./IQA Value.xlsx`
- `CFG["DATA_DIR"]` and `CFG["ANNOT_CSV"]` are set to these paths automatically.


In [ ]:
import zipfile, shutil
from pathlib import Path

ZIP_NAME = "UWIQA Dataset.zip"      # zip file sitting next to this notebook
ZIP_PATH = Path(ZIP_NAME)

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Could not find '{ZIP_NAME}' in the current directory ({Path.cwd()}).\n"
        "Please place 'UWIQA Dataset.zip' next to this notebook and re-run."
    )

print(f"Found: {ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")

# Extract
EXTRACT_DIR = Path(".")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    members = zf.namelist()
    print(f"Contents ({len(members)} entries): {members[:5]} ...")
    zf.extractall(EXTRACT_DIR)

print("Extraction complete.")

# ── Locate the image folder and IQA Value.xlsx inside the extracted tree ──
# Handles both: zip root = "UWIQA Dataset/" OR files at root level
def find_in_extracted(filename_or_folder, start=EXTRACT_DIR):
    """Walk extracted tree to find a file or folder by name."""
    for p in Path(start).rglob("*"):
        if p.name == filename_or_folder:
            return p
    return None

img_dir  = find_in_extracted("UWIQA")
mos_file = find_in_extracted("IQA Value.xlsx")

if img_dir is None:
    raise FileNotFoundError(
        "Could not find 'UWIQA' folder in the extracted zip. "
        f"Extracted entries: {[str(p) for p in Path(EXTRACT_DIR).iterdir()]}"
    )
if mos_file is None:
    raise FileNotFoundError(
        "Could not find 'IQA Value.xlsx' in the extracted zip."
    )

print(f"Image folder : {img_dir}")
print(f"IQA Value file : {mos_file}")

# ── Count images to confirm ───────────────────────────────────────────────
IMAGE_EXTS_CHECK = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
img_count = len([p for p in img_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS_CHECK])
print(f"Images found : {img_count}")

# ── Expose paths as module-level variables (used by CFG below) ────────────
DATASET_IMAGE_DIR = str(img_dir)
DATASET_MOS_PATH  = str(mos_file)

print(f"\nv Dataset ready.")
print(f"  DATASET_IMAGE_DIR = {DATASET_IMAGE_DIR!r}")
print(f"  DATASET_MOS_PATH  = {DATASET_MOS_PATH!r}")


## 1 . Research Objective

Underwater images suffer from *spatially non-uniform* degradations caused by wavelength-dependent light absorption, forward/backward scattering, turbidity, non-uniform illumination, and suspended particles. A conventional full-image quality metric ignores this spatial heterogeneity.

**Goal:** learn both *local patch quality* $q_i$ and *perceptual importance* $w_i$ for every spatial patch, then aggregate:

$$Q = \frac{\sum_{i=1}^{N} w_i q_i}{\sum_{i=1}^{N} w_i}$$

This notebook implements the **hybrid Solution III + Solution II** architecture described in the design document:

* **Solution III (backbone):** Multi-Scale Patch Extraction -> CNN Feature Extraction -> Transformer Cross-Patch Context Modeling -> Hierarchical Importance Pooling  
* **Solution II (head):** Three parallel branches (Quality / Saliency / Degradation) applied to each Transformer-contextualized patch feature $h_i$

---


## 2 . Architecture Overview

```
Input Underwater Image  (H x W x 3)
        |
        v
Multi-Scale Patch Extraction
  +-- Scale 1: 32x32 patches  -> N1 = (H/32).(W/32) patches
  +-- Scale 2: 64x64 patches  -> N2 = (H/64).(W/64) patches
  +-- Scale 3: 128x128 patches-> N3 = (H/128).(W/128) patches
        |
        v  (for each scale independently)
CNN Patch Feature Extractor  (shared or scale-specific backbone)
  fi(s) = F(Pi(s))  in Rd
        |
        v
Scale Embedding Injection
  token = fi(s) + scale_embed(s)
        |
        v
Transformer Encoder  (L layers, multi-head self-attention)
  X -> X' = Transformer(X)
  hi = contextualized patch feature  in Rd
        |
        v  (three parallel branches per patch)
+-------------------------------------------------------+
|  Quality Branch   qi = sigma(Wq hi + bq)                |
|  Saliency Branch  si = sigma(Ws hi + bs)                |
|  Degradation Br.  ri = [color,haze,blur,contrast,noise]|
+-------------------------------------------------------+
        |
        v
Importance Fusion
  ei = G(hi, qi, si, di)
  alphai = softmax(ei)
        |
        v
Scale-Adaptive Hierarchical Pooling
  Qs = Sigmai alphai(s) qi(s)
  Q  = Sigmas lambdas Qs   where Sigmalambdas = 1
        |
        v
Global Underwater Image Quality Score  Q in [0, 1]
```

**Why multi-scale patches are necessary:**
- **32x32 (fine scale):** captures *local* degradation - colour cast in a small region, blur in a texture patch, noise in a dark corner.  
- **64x64 (medium scale):** captures *mid-range structure* - fin of a fish, coral branch. Balances local/global.  
- **128x128 (coarse scale):** captures *semantic context* - is this patch background water or a foreground subject? A severe degradation on a semantically irrelevant background should contribute less.  

Without multi-scale representation, a single patch size either misses fine local artefacts (too large) or lacks semantic context to weight patches correctly (too small).

---


## 3 . Mathematical Formulation

### 3.1 Patch Extraction

For image $\mathbf{I} \in \mathbb{R}^{H \times W \times 3}$ and patch size $p$:

$$N_s = \left\lfloor \frac{H}{p_s} \right\rfloor \cdot \left\lfloor \frac{W}{p_s} \right\rfloor, \quad \mathcal{P}^{(s)} = \{P_1^{(s)}, \ldots, P_{N_s}^{(s)}\}$$

### 3.2 CNN Feature Extraction

$$\mathbf{f}_i^{(s)} = F_\theta(P_i^{(s)}), \quad \mathbf{f}_i^{(s)} \in \mathbb{R}^d$$

### 3.3 Transformer Self-Attention

$$\mathbf{Q} = \mathbf{X}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}_V$$

$$\mathbf{A} = \text{Softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right), \quad \mathbf{X}' = \mathbf{A}\mathbf{V}$$

### 3.4 Three-Branch Head

$$q_i = \sigma(W_q h_i + b_q), \quad q_i \in [0,1]$$

$$s_i = \sigma(W_s h_i + b_s), \quad s_i \in [0,1]$$

$$\mathbf{r}_i = [r_i^{\text{color}}, r_i^{\text{haze}}, r_i^{\text{blur}}, r_i^{\text{contrast}}, r_i^{\text{noise}}], \quad d_i = \sum_k \beta_k r_i^{(k)}$$

### 3.5 Importance Fusion

$$e_i = G(h_i, q_i, s_i, d_i), \quad \alpha_i = \frac{\exp(e_i)}{\sum_j \exp(e_j)}$$

### 3.6 Scale-Adaptive Pooling

$$Q^{(s)} = \sum_i \alpha_i^{(s)} q_i^{(s)}, \quad \hat{Q} = \sum_s \lambda_s Q^{(s)}, \quad \sum_s \lambda_s = 1$$

### 3.7 Loss Functions

$$\mathcal{L}_{\text{MOS}} = \frac{1}{M}\sum_n (\hat{y}_n - y_n)^2$$

$$\mathcal{L}_{\text{rank}} = \max\!\left(0,\; m - (\hat{y}_a - \hat{y}_b)\right) \quad \text{for } y_a > y_b$$

$$\mathcal{L} = \lambda_{\text{MOS}}\,\mathcal{L}_{\text{MOS}} + \lambda_{\text{rank}}\,\mathcal{L}_{\text{rank}} + \lambda_{\text{deg}}\,\mathcal{L}_{\text{deg}}$$

$\mathcal{L}_{\text{deg}}$ is **automatically disabled** if no degradation labels exist.

### 3.8 Explainability Maps

$$c_i = \alpha_i q_i \quad \text{(quality contribution)}$$

$$r_i = \alpha_i (1-q_i) \quad \text{(degradation contribution)}$$

> **Interpretation note:** These maps reflect *model attributions*, not physical ground-truth degradation measurements.

---


## 4 . Environment Setup

Install / verify required packages.

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Core requirements
pip_install(
    "torch>=2.0",
    "torchvision>=0.15",
    "timm>=0.9.12",   # DINOv2 weights in timm
    "Pillow",
    "pandas",
    "numpy",
    "matplotlib",
    "scipy",
    "scikit-learn",
    "tqdm",
)
print("v All packages installed / confirmed.")


In [ ]:
import os, random, math, json, warnings
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm
from torch.cuda.amp import autocast, GradScaler

from scipy.stats import pearsonr, spearmanr, kendalltau
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")


## 5 . Configuration

All hyper-parameters, paths and architectural choices live in a single `CFG` dictionary. Modify only here.

**Implementation decisions (not from the design document):**
- `IMAGE_SIZE = 256` - resize to square before patch extraction (required to guarantee exact patch counts).  
- `CNN_BACKBONE = "resnet18"` - lightweight for prototyping; swap to `"resnet50"` for full experiments.  
- `EMBED_DIM = 256` - feature dimensionality throughout.  


In [ ]:
CFG = {
    # -- Paths --------------------------------------------------------------
    "DATA_DIR"        : DATASET_IMAGE_DIR,     # set by unzip cell above
    "ANNOT_CSV"       : DATASET_MOS_PATH,      # set by unzip cell above
    "CHECKPOINT_DIR"  : "./checkpoints",
    "OUTPUT_DIR"      : "./outputs",

    # -- Reproducibility ----------------------------------------------------
    "SEED"            : 42,                # fixes the data split AND the k-fold partition (keep fixed)
    "SEEDS"           : [42],              # single seed for now (add 43, 44 later for mean +/- std)
    "RUN_MAIN_EXPERIMENT": True,           # seeds x k-fold run in the summary cell

    # -- Image preprocessing ------------------------------------------------
    "IMAGE_SIZE"      : 256,   # resize to IMAGE_SIZE x IMAGE_SIZE (square). 384 = ablation (divisible by 32/64/128)

    # -- Multi-scale patch sizes --------------------------------------------
    "PATCH_SIZES"     : [32, 64, 128],
    # For IMAGE_SIZE=256:  N = 64 + 16 + 4 = 84 tokens. For 384: 144 + 36 + 9 = 189.

    # -- CNN / ViT backbone -------------------------------------------------
    "CNN_BACKBONE"    : "resnet50",   # "resnet18" | "resnet50" | "dinov2_vits14" | "dinov2_vitb14" | "swin_t" | "swin_s"
    "CNN_PRETRAINED"  : True,
    "CNN_INPUT_SIZE"  : 224,          # ViT backbones only: DINOv2 patch size (multiple of 14); Swin whole-image size (must be 224)
    "EMBED_DIM"       : 256,          # projected feature dimension d

    # -- Transformer --------------------------------------------------------
    "TRANSFORMER_DEPTH"  : 4,
    "TRANSFORMER_HEADS"  : 8,
    "TRANSFORMER_FF_DIM" : 512,
    "TRANSFORMER_DROPOUT": 0.4,

    # -- Degradation branch -------------------------------------------------
    "N_DEGRADATION_ATTRS": 5,     # [color, haze, blur, contrast, noise]
    # True = handcrafted per-patch pseudo-labels (compute_degradation_pseudolabels) supervise r_i.
    # They are NOT human ground truth. Set False to drop them.
    "HAS_DEGRADATION_GT" : True,

    # -- Degradation estimator + FiLM feedback (v3) -------------------------
    "USE_FILM"           : True,   # estimator embedding z FiLM-conditions the Transformer tokens
    "DEG_EMBED_DIM"      : 128,
    "DEG_NORM"           : "group", # degradation estimator norm: "group" (GroupNorm, batch-size independent) | "batch"
    "USE_DEG_PRETRAIN"   : True,   # stage 1: label-free synthetic-degradation pretraining
    "DEG_PRETRAIN_EPOCHS": 8,
    "DEG_PRETRAIN_ITERS" : 100,    # iterations per epoch
    "DEG_PRETRAIN_LR"    : 3e-4,

    # -- Training -----------------------------------------------------------
    # -- Class balancing (v3b) ----------------------------------------------
    # Classes = MOS bins of width CLASS_BIN_WIDTH. Only the TRAIN loader is balanced (val/test keep the
    # natural distribution). Minority classes are re-drawn more often, and every draw gets fresh random
    # label-preserving augmentation. BALANCE_POWER: 0 = natural sampling, 1 = fully balanced, 0.5 = in between.
    "BALANCE_CLASSES" : True,
    "CLASS_BIN_WIDTH" : 0.1,
    "BALANCE_POWER"   : 1.0,
    "BALANCE_ROT90"   : True,   # extra label-preserving augmentation (90/180/270 deg) for the train loader

    "BATCH_SIZE"      : 16,
    "NUM_EPOCHS"      : 200,
    "LR"              : 3e-5,
    "WEIGHT_DECAY"    : 1e-3,
    "BACKBONE_LR_MULT": 0.2,   # backbone LR = LR * this once unfrozen (heads keep LR)
    "EMA_DECAY"       : 0.995, # validation / checkpoints use the EMA weights
    "GRAD_CLIP"       : 1.0,
    "EARLY_STOP_PAT"  : 30,    # an iteration ends after 30 epochs without a val-SROCC gain (counted from the backbone unfreeze)
    "BATCHES_PER_EPOCH" : 25,    # batches per epoch (the "iterations per epoch" setting; samples per epoch = this x BATCH_SIZE)
    "SAVE_EPOCH_CKPTS": False, # True also writes one .pt per epoch (large: a full model every epoch)
    "FREEZE_EPOCHS"   : 20,    # pretrained backbone frozen for the first N epochs (0 = never)

    # -- Architecture add-ons -----------------------------------------------
    "USE_SE"          : True,  # SE channel attention after the CNN (ResNet only)

    # -- Loss terms ---------------------------------------------------------
    "MOS_LOSS"          : "huber",  # "huber" (v3 default) | "mse" (v2)
    "HUBER_DELTA"       : 0.1,      # in MOS units; residuals beyond this are penalised linearly. Tune to the MOS scale.
    "LAMBDA_PLCC"       : 0.5,      # weight of (1 - batch PLCC) loss; 0 disables
    "RANK_WEIGHT_GAMMA" : 0.5,      # up-weights extreme-MOS samples in the MOS loss; 0 disables

    # -- Test-time augmentation ---------------------------------------------
    "USE_TTA"         : True,
    "TTA_CROP"        : False,      # crop views lowered SROCC earlier; flips only

    # -- Optional KADID-10k pretraining (set the path to enable) ------------
    "KADID_DIR"       : None,   # e.g. "/content/kadid10k" (needs images/ and dmos.csv)
    "KADID_CSV"       : None,   # default: <KADID_DIR>/dmos.csv
    "USE_KADID_INIT"  : True,   # False = ignore a cached KADID checkpoint (ImageNet-only baseline); used by the KADID check
    "PRETRAIN_EPOCHS" : 30,
    "PRETRAIN_ITERS"  : 100,    # iterations per pretraining epoch
    "PRETRAIN_LR"     : 1e-4,

    # -- K-fold ensemble (v3: on by default) --------------------------------
    # Each seed trains KFOLD models on folds of the train+val pool; the test split stays held out.
    # Test predictions = mean over fold models. 0 or 1 = single train/val run.
    "KFOLD"           : 10,  # 10 iterations: one full train-until-patience-then-evaluate run per fold

    # -- Calibration --------------------------------------------------------
    # 5-parameter logistic fitted on out-of-fold validation predictions, applied to test.
    # (v2 fitted it on TRAIN predictions, which are in-sample and optimistic.)

    # -- Loss weights -------------------------------------------------------
    "LAMBDA_MOS"      : 1.0,
    "LAMBDA_RANK"     : 0.5,
    "LAMBDA_DEG"      : 0.1,   # used only when HAS_DEGRADATION_GT is True
    "RANK_MARGIN"     : 0.05,

    # -- Dataset split ------------------------------------------------------
    # 70/15/15 is NOT verified against EDANet's protocol (see the changelog cell).
    "TRAIN_RATIO"     : 0.70,
    "VAL_RATIO"       : 0.10,
    "TEST_RATIO"      : 0.20,   # must sum to 1.0

    # -- Mixed precision ----------------------------------------------------
    "USE_AMP"         : True,
}

# Validate split ratios
assert abs(CFG["TRAIN_RATIO"]+CFG["VAL_RATIO"]+CFG["TEST_RATIO"]-1.0) < 1e-6, \
    "Split ratios must sum to 1.0"
assert CFG["CNN_INPUT_SIZE"] % 14 == 0 or not CFG["CNN_BACKBONE"].startswith("dinov2"), \
    "CNN_INPUT_SIZE must be a multiple of 14 for DINOv2"
assert CFG["CNN_INPUT_SIZE"] == 224 or not CFG["CNN_BACKBONE"].startswith("swin"), \
    "CNN_INPUT_SIZE must be 224 for the pretrained Swin backbones"

for d in [CFG["CHECKPOINT_DIR"], CFG["OUTPUT_DIR"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"Backbone       : {CFG['CNN_BACKBONE']}")
print(f"Patch sizes    : {CFG['PATCH_SIZES']}")
print(f"For {CFG['IMAGE_SIZE']}x{CFG['IMAGE_SIZE']} image:")
for p in CFG["PATCH_SIZES"]:
    n = (CFG["IMAGE_SIZE"]//p)**2
    print(f"  {p:3d}x{p:3d} -> {n:3d} patches")
print(f"  Total tokens : {sum((CFG['IMAGE_SIZE']//p)**2 for p in CFG['PATCH_SIZES'])}")
print(f"MOS loss       : {CFG['MOS_LOSS']} (delta={CFG['HUBER_DELTA']})")
print(f"K-fold         : {CFG['KFOLD']}   seeds: {CFG['SEEDS']}")
print(f"FiLM / deg-pre : {CFG['USE_FILM']} / {CFG['USE_DEG_PRETRAIN']}")


## 6 . Reproducibility

Fix all random seeds *before* any data operations.

In [ ]:
def set_seed(seed: int) -> None:
    '''Fix random state for reproducibility across Python, NumPy and PyTorch.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG["SEED"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seed fixed: {CFG['SEED']}")
print(f"Device    : {DEVICE}")


## 7 . Dataset Discovery

The code scans `CFG["DATA_DIR"]` for images.

**Label policy:**
- If `CFG["ANNOT_CSV"]` exists (your `IQA Value.xlsx`), quality labels are loaded and merged.
- If the file is missing, the pipeline still supports patch extraction, inference, and visualization, but **supervised training and PLCC/SROCC/RMSE evaluation require quality labels**.
- Labels are **never fabricated**.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def discover_dataset(data_dir: str, annot_path_str: Optional[str]) -> pd.DataFrame:
    '''
    Scans data_dir for images (e.g. 0001.png, 0002.png, ...) and merges quality
    labels from an Excel or CSV file.

    Expected Excel/CSV columns (your dataset):
      "Image"      - filename, e.g. 0001.png  (with extension, exact match)
      "IQA Value"  - float quality score

    Returns a DataFrame with columns [image_path, image_name, mos].
    Does NOT fabricate labels.
    '''
    data_dir = Path(data_dir)
    if not data_dir.exists():
        print(f"WARNING  Image folder '{data_dir}' not found.")
        print("   Please place your 'UWIQA' folder next to this notebook.")
        print("   Creating a synthetic demo dataset so cells below can be tested.")
        data_dir.mkdir(parents=True, exist_ok=True)
        _make_demo_images(data_dir)

    paths = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS])
    if not paths:
        raise FileNotFoundError(f"No images found in {data_dir}")

    df = pd.DataFrame({"image_path": [str(p) for p in paths]})
    df["image_name"] = [Path(p).name for p in paths]

    # ── Load quality labels from Excel or CSV ──────────────────────────
    annot_path = Path(annot_path_str) if annot_path_str else None
    has_labels = annot_path is not None and annot_path.exists()

    if has_labels:
        ext = annot_path.suffix.lower()
        if ext in (".xlsx", ".xls"):
            ann = pd.read_excel(annot_path)
        elif ext == ".csv":
            ann = pd.read_csv(annot_path)
        else:
            raise ValueError(f"Unsupported annotation format: {ext}. Use .xlsx or .csv")

        # ── Validate and normalise columns ─────────────────────────────
        print(f"Annotation file columns: {list(ann.columns)}")

        # Accept your exact column names OR common alternatives
        col_image = None
        col_mos   = None
        for c in ann.columns:
            cl = c.strip().lower()
            if cl in ("image", "filename", "file", "image_name", "underwater image", "underwater_image"):
                col_image = c
            if cl in ("iqa value", "iqa_value", "mos", "quality", "score", "mean opinion score", "mos_score", "mos score"):
                col_mos = c

        if col_image is None or col_mos is None:
            raise ValueError(
                f"Could not identify image/quality-value columns.\n"
                f"Found columns: {list(ann.columns)}\n"
                f"Expected: 'Image' and 'IQA Value' (or common variants)."
            )

        print(f"Using columns: image='{col_image}'  mos='{col_mos}'")

        ann = ann[[col_image, col_mos]].copy()
        ann.columns = ["image_name", "mos"]
        ann["image_name"] = ann["image_name"].astype(str).str.strip()

        # --- Format image_name to match 0001.png format --- #
        # Assuming image names are numbers that need leading zeros and a .png extension
        # This ensures '1' becomes '0001.png' to match discovered image files
        try:
            ann["image_name"] = ann["image_name"].apply(lambda x: f"{int(x):04d}.png")
        except ValueError:
            print("WARNING: Could not convert image names to 4-digit numbers with .png extension. Ensure image names in Excel are numeric.")

        ann["mos"]        = pd.to_numeric(ann["mos"], errors="coerce")

        # ── Merge on filename ──────────────────────────────────────────
        df = df.merge(ann, on="image_name", how="left")

        n_matched = df["mos"].notna().sum()
        n_missing = df["mos"].isna().sum()
        print(f"v Quality labels: {n_matched}/{len(df)} images matched.")
        if n_missing > 0:
            print(f"  WARNING: {n_missing} images have no IQA Value entry in the spreadsheet.")
            print(f"  Unmatched sample: {df.loc[df['mos'].isna(), 'image_name'].head(3).tolist()}")

    else:
        df["mos"] = float("nan")
        if annot_path:
            print(f"WARNING  Annotation file '{annot_path}' not found.")
        else:
            print("WARNING  No annotation file configured.")
        print("   Inference and visualisation will work; supervised training will not.")

    print(f"Dataset: {len(df)} images  |  folder: '{data_dir}'")
    if "mos" in df.columns and df["mos"].notna().any():
        print(f"MOS range: [{df['mos'].min():.3f}, {df['mos'].max():.3f}]  "
              f"mean={df['mos'].mean():.3f}  std={df['mos'].std():.3f}")
    return df

def _make_demo_images(directory: Path, n: int = 6) -> None:
    '''Create small synthetic RGB images for smoke-testing only.'''
    for i in range(n):
        arr = np.random.randint(0, 180, (256, 256, 3), dtype=np.uint8)
        arr[:, :, 0] = arr[:, :, 0] // 3   # blue-green cast
        Image.fromarray(arr).save(directory / f"{i+1:04d}.png")
    print(f"  Created {n} synthetic demo images in '{directory}'.")


df_all = discover_dataset(CFG["DATA_DIR"], CFG["ANNOT_CSV"])
df_all.head(10)

## 8 . Image-Level Train / Validation / Test Split

**Critical data-leakage prevention:**  
The split is performed at the **image level** before any patch extraction. Patches from the same source image will never appear in more than one split. A fixed seed guarantees reproducibility.


In [ ]:
from sklearn.model_selection import StratifiedKFold


def strat_labels(mos, min_count: int, width: float = None) -> np.ndarray:
    '''MOS bins (width CLASS_BIN_WIDTH) used only for stratification. Bins with fewer than
    min_count images are merged into the nearest bin so every stratum can be split.'''
    width = width or CFG["CLASS_BIN_WIDTH"]
    mos = np.asarray(mos, dtype=float)
    lab = np.where(np.isnan(mos), -999, np.round(np.nan_to_num(mos) / width)).astype(int)
    while len(np.unique(lab)) > 1:
        vals, cnt = np.unique(lab, return_counts=True)
        j = int(np.argmin(cnt))
        if cnt[j] >= min_count:
            break
        others = np.delete(vals, j)
        lab[lab == vals[j]] = others[np.argmin(np.abs(others - vals[j]))]
    return lab


def make_folds(pool: pd.DataFrame, k: int, seed: int):
    '''Stratified k-fold on MOS bins. Returns a list of (train_idx, val_idx) positional index arrays.'''
    y = strat_labels(pool["mos"].values, k)
    return list(StratifiedKFold(n_splits=k, shuffle=True, random_state=seed).split(pool, y))


def split_dataset(
    df: pd.DataFrame,
    train_ratio: float,
    val_ratio: float,
    seed: int
) -> pd.DataFrame:
    '''
    Splits at the image level. Returns df with a new 'split' column.
    No patch from the same image appears in more than one split.
    '''
    indices = df.index.tolist()
    test_ratio = 1.0 - train_ratio - val_ratio

    # Stratified by MOS bin so train / val / test cover the same quality range
    try:
        train_idx, temp_idx = train_test_split(
            indices, test_size=(1.0 - train_ratio), random_state=seed, shuffle=True,
            stratify=strat_labels(df["mos"].values, 20)
        )
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=test_ratio / (val_ratio + test_ratio),
            random_state=seed, shuffle=True,
            stratify=strat_labels(df.loc[temp_idx, "mos"].values, 4)
        )
    except ValueError as e:
        print(f"WARNING: stratified split failed ({e}); falling back to a random split.")
        train_idx, temp_idx = train_test_split(
            indices, test_size=(1.0 - train_ratio), random_state=seed, shuffle=True
        )
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=test_ratio / (val_ratio + test_ratio),
            random_state=seed, shuffle=True
        )

    df = df.copy()
    df["split"] = "train"
    df.loc[val_idx, "split"] = "val"
    df.loc[test_idx, "split"] = "test"
    return df


df_all = split_dataset(
    df_all,
    CFG["TRAIN_RATIO"], CFG["VAL_RATIO"], CFG["SEED"]
)

print("Split summary:")
print(df_all["split"].value_counts().to_string())
print("\nMean MOS by split (should be close):")
print(df_all[df_all["mos"].notna()].groupby("split")["mos"].agg(["count", "mean", "std"]).round(3).to_string())
print(f"\nSeed used: {CFG['SEED']}  (document and reproduce this)")

# Verify no overlap
for a, b in [("train","val"), ("train","test"), ("val","test")]:
    s1 = set(df_all.loc[df_all["split"]==a, "image_path"])
    s2 = set(df_all.loc[df_all["split"]==b, "image_path"])
    assert s1.isdisjoint(s2), f"ERROR Data leakage detected between {a} and {b}!"
print("v No data leakage between splits.")

## 8b · Class distribution and class-balanced augmentation (v3b)

Classes are MOS bins. Before training, the bar graphs show the natural class counts and the images drawn per epoch once the train loader is class-balanced. Only the training loader is balanced and augmented; validation and test stay natural, so the reported metrics are not distorted. Augmentations are label-preserving (flips, 90-degree rotations, mild brightness/contrast jitter, <=10% crop). Hue and saturation jitter and blur are excluded because they change the quality label.


In [ ]:
# ---- Class definition, balancing weights and the "before / after" bar graph (v3b) ----
from sklearn.model_selection import KFold

def class_ids(mos: np.ndarray, cfg: dict) -> np.ndarray:
    """Class = MOS bin of width CLASS_BIN_WIDTH (0.1 -> classes 0.2, 0.3, ... for this dataset)."""
    return np.round(np.asarray(mos, dtype=float) / cfg["CLASS_BIN_WIDTH"]).astype(int)

def class_balance_weights(mos: np.ndarray, cfg: dict) -> np.ndarray:
    """Per-sample draw weight = (class count) ** -BALANCE_POWER. Power 1 gives equal expected draws per class."""
    cid = class_ids(mos, cfg)
    vals, counts = np.unique(cid, return_counts=True)
    cnt = dict(zip(vals, counts))
    return np.array([cnt[c] ** (-cfg.get("BALANCE_POWER", 1.0)) for c in cid], dtype=float)

def plot_class_distribution(df: pd.DataFrame, cfg: dict, n_epochs_sim: int = 200) -> None:
    lab = df[df["mos"].notna()].copy()
    lab["cls"] = class_ids(lab["mos"].values, cfg)
    classes = sorted(lab["cls"].unique())
    names = [f"{c * cfg['CLASS_BIN_WIDTH']:.1f}" for c in classes]

    # Training set of fold 1 (what the model actually trains on); fall back to the train split
    pool = lab[lab["split"].isin(["train", "val"])].reset_index(drop=True)
    if cfg.get("KFOLD", 0) >= 2:
        tr_idx, _ = make_folds(pool, cfg["KFOLD"], cfg["SEED"])[0]
        tr = pool.iloc[tr_idx]
    else:
        tr = lab[lab["split"] == "train"]
    nat = tr["cls"].value_counts().reindex(classes, fill_value=0).values

    # Per-epoch draws of the balanced sampler, simulated over many epochs (no images are loaded)
    n_draw = cfg.get("BATCHES_PER_EPOCH", 25) * cfg["BATCH_SIZE"]
    w = class_balance_weights(tr["mos"].values, cfg)
    rng = np.random.default_rng(0)
    draws = rng.choice(len(tr), size=n_epochs_sim * n_draw, replace=True, p=w / w.sum())
    bal = pd.Series(tr["cls"].values[draws]).value_counts().reindex(classes, fill_value=0).values / n_epochs_sim

    fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
    # (1) whole dataset, stacked by split
    bottom = np.zeros(len(classes))
    for sp, col in [("train", "#4C78A8"), ("val", "#F58518"), ("test", "#54A24B")]:
        v = lab[lab["split"] == sp]["cls"].value_counts().reindex(classes, fill_value=0).values
        ax[0].bar(names, v, bottom=bottom, label=sp, color=col); bottom += v
    for x, t in zip(names, bottom): ax[0].text(x, t, int(t), ha="center", va="bottom", fontsize=8)
    ax[0].set_title("All labelled images, by split"); ax[0].legend()
    # (2) before balancing
    b = ax[1].bar(names, nat, color="#4C78A8"); ax[1].bar_label(b, fontsize=8)
    ax[1].set_title("Training set BEFORE balancing (natural)")
    # (3) after balancing
    b = ax[2].bar(names, bal, color="#E45756"); ax[2].bar_label(b, fmt="%.0f", fontsize=8)
    ax[2].set_title(f"Images drawn per epoch AFTER balancing (power={cfg['BALANCE_POWER']})")
    for a in ax: a.set_xlabel("MOS class (bin width %.1f)" % cfg["CLASS_BIN_WIDTH"]); a.set_ylabel("images")
    plt.tight_layout()
    out = Path(cfg["OUTPUT_DIR"]) / "class_distribution.png"
    plt.savefig(out, dpi=150); plt.show()
    print(pd.DataFrame({"MOS class": names, "train (natural)": nat,
                        "drawn / epoch (balanced)": np.round(bal, 1),
                        "max repeats of one image / epoch": np.round(bal / np.maximum(nat, 1), 1)}).to_string(index=False))
    print(f"\nSaved {out}")

if CFG.get("BALANCE_CLASSES", False):
    plot_class_distribution(df_all, CFG)
else:
    print("BALANCE_CLASSES is False: training uses natural sampling.")


## 8c · Data sanity checks: MOS direction and near-duplicate images

1. **MOS direction.** The ranking loss assumes a larger MOS means better quality. The grid shows the lowest- and highest-MOS images so you can confirm this by eye; the two simple image statistics are only a weak supporting hint.
2. **Near-duplicates.** Near-identical images in different splits make validation or test optimistic. The check lists candidate pairs (very similar tiny greyscale thumbnails), which you should inspect before trusting them as true duplicates.


In [ ]:
def check_mos_direction(n_show: int = 4, n_proxy: int = 200, seed: int = 0) -> None:
    lab = df_all[df_all["mos"].notna()].sort_values("mos").reset_index(drop=True)
    if len(lab) < 2 * n_show:
        print("Not enough labelled images for the direction check."); return
    picks = pd.concat([lab.head(n_show), lab.tail(n_show)])
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8.4))
    for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
        ax.imshow(Image.open(r["image_path"]).convert("RGB")); ax.axis("off")
        ax.set_title(f"{r['image_name']}  MOS={r['mos']:.2f}", fontsize=9)
    fig.suptitle("Top row: LOWEST MOS    |    Bottom row: HIGHEST MOS    (higher MOS should look better)", fontsize=11)
    plt.tight_layout()
    plt.savefig(Path(CFG["OUTPUT_DIR"]) / "mos_direction_check.png", dpi=110); plt.show()

    s = lab.sample(min(n_proxy, len(lab)), random_state=seed)
    sharp, contrast = [], []
    for p in s["image_path"]:
        g = np.asarray(Image.open(p).convert("L").resize((256, 256)), dtype=np.float32) / 255.0
        sharp.append(np.abs(np.diff(g, axis=1)).mean() + np.abs(np.diff(g, axis=0)).mean())
        contrast.append(g.std())
    print("Spearman correlation of MOS with simple image statistics (weak hints only, scene content also matters):")
    for name, v in (("sharpness (mean abs gradient)", sharp), ("contrast (grey std)", contrast)):
        print(f"  {name:32s}: {spearmanr(s['mos'], v)[0]:+.3f}")
    print("Positive values and a better-looking bottom row support 'higher MOS = better quality'. "
          "If the top row looks better, the labels are a distortion score and need to be flipped or described accordingly.")


def near_duplicate_check(thresh: float = 0.995, size: int = 16) -> pd.DataFrame:
    names = df_all["image_name"].values
    splits = df_all["split"].values
    mos = df_all["mos"].values
    X = np.stack([np.asarray(Image.open(p).convert("L").resize((size, size)), dtype=np.float32).ravel()
                  for p in df_all["image_path"]])
    X -= X.mean(axis=1, keepdims=True)
    X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    S = X @ X.T
    iu = np.triu_indices(len(names), 1)
    keep = S[iu] > thresh
    ii, jj = iu[0][keep], iu[1][keep]
    pairs = pd.DataFrame({"image_a": names[ii], "image_b": names[jj], "split_a": splits[ii], "split_b": splits[jj],
                          "mos_a": mos[ii], "mos_b": mos[jj], "similarity": S[ii, jj]})
    pairs["kind"] = ["-".join(sorted([a, b])) for a, b in zip(pairs["split_a"], pairs["split_b"])]
    pairs = pairs.sort_values("similarity", ascending=False)
    print(f"Candidate near-duplicate pairs (thumbnail similarity > {thresh}): {len(pairs)}")
    if len(pairs):
        print(pairs["kind"].value_counts().to_string())
        print("\nMost similar pairs:"); print(pairs.head(10).round(3).to_string(index=False))
    else:
        print("None found at this threshold.")
    pairs.to_csv(Path(CFG["OUTPUT_DIR"]) / "near_duplicate_candidates.csv", index=False)
    return pairs

if HAS_LABELS if "HAS_LABELS" in globals() else df_all["mos"].notna().any():
    check_mos_direction()
    DUP_PAIRS = near_duplicate_check()
else:
    print("No MOS labels: direction / duplicate checks skipped.")


## 9 . Multi-Scale Patch Extraction

### Why this approach

`torch.nn.functional.unfold` performs the patch extraction as a **single vectorized operation** on the GPU. This is substantially faster than Python loops and avoids CPU-GPU data transfer overhead.

**Input shape:** `(B, C, H, W)`  
**After unfold for patch size $p$:** `(B, C.p.p, N_patches)`  
**Reshaped per patch:** `(B, N, C, p, p)`

### Verified patch counts for `IMAGE_SIZE = 256`

| Scale | Patch size | N patches |
|-------|-----------|-----------|
| 1 | 32 x 32 | 64 |
| 2 | 64 x 64 | 16 |
| 3 | 128 x 128 | 4 |
| **Total** | | **84** |


In [ ]:
def extract_patches_multiscale(
    images: torch.Tensor,
    patch_sizes: List[int]
) -> Dict[int, torch.Tensor]:
    '''
    Vectorized multi-scale patch extraction using torch.nn.functional.unfold.

    Args:
        images : (B, 3, H, W) - batch of images, values in [0,1]
        patch_sizes : list of int patch sizes

    Returns:
        dict mapping patch_size -> tensor of shape (B, N_patches, 3, p, p)
    '''
    B, C, H, W = images.shape
    result: Dict[int, torch.Tensor] = {}

    for p in patch_sizes:
        assert H % p == 0 and W % p == 0, \
            f"Image size {H}x{W} not divisible by patch size {p}"

        # unfold: (B, C*p*p, N_patches)
        patches_flat = F.unfold(images, kernel_size=p, stride=p)
        N = patches_flat.shape[-1]

        # reshape to (B, N, C, p, p)
        patches = patches_flat.permute(0, 2, 1)          # (B, N, C*p*p)
        patches = patches.reshape(B, N, C, p, p)         # (B, N, 3, p, p)
        result[p] = patches

    return result


# -- Sanity check -------------------------------------------------------------
print("Verifying patch extraction shapes:")
_dummy = torch.zeros(2, 3, CFG["IMAGE_SIZE"], CFG["IMAGE_SIZE"])
_patches = extract_patches_multiscale(_dummy, CFG["PATCH_SIZES"])
expected_total = 0
for p in CFG["PATCH_SIZES"]:
    t = _patches[p]
    N_expected = (CFG["IMAGE_SIZE"] // p) ** 2
    assert t.shape == (2, N_expected, 3, p, p), \
        f"Shape mismatch for p={p}: got {t.shape}"
    expected_total += N_expected
    print(f"  Patch size {p:3d}x{p:3d} -> shape {tuple(t.shape)}  (N={N_expected})")
assert expected_total == 84, f"Expected 84 total tokens, got {expected_total}"
print(f"  Total tokens per image : {expected_total}  v")


## 10 . Patch Visualization

Visualize the multi-scale patch grid on a sample image.

In [ ]:
def visualize_patches(image_path: str, patch_sizes: List[int], img_size: int) -> None:
    '''Draw patch grids over a sample image at each scale.'''
    img = Image.open(image_path).convert("RGB").resize((img_size, img_size))
    img_arr = np.array(img)

    fig, axes = plt.subplots(1, len(patch_sizes)+1, figsize=(5*(len(patch_sizes)+1), 5))
    axes[0].imshow(img_arr); axes[0].set_title("Original (resized)"); axes[0].axis("off")

    colors = ["red", "lime", "cyan"]
    for ax, p, col in zip(axes[1:], patch_sizes, colors):
        ax.imshow(img_arr)
        n = img_size // p
        for r in range(n):
            for c in range(n):
                rect = mpatches.Rectangle(
                    (c*p, r*p), p, p,
                    linewidth=1, edgecolor=col, facecolor="none"
                )
                ax.add_patch(rect)
        N = n*n
        ax.set_title(f"{p}x{p} patches  (N={N})")
        ax.axis("off")

    plt.suptitle("Multi-Scale Patch Grids", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(Path(CFG["OUTPUT_DIR"]) / "patch_grids.png", dpi=120)
    plt.show()
    print("Patch grid saved to outputs/patch_grids.png")


# Run on the first available image
_sample_path = df_all["image_path"].iloc[0]
visualize_patches(_sample_path, CFG["PATCH_SIZES"], CFG["IMAGE_SIZE"])


## 11 . Dataset Class

The `UWIQADataset` class:
- Loads and resizes images to `IMAGE_SIZE x IMAGE_SIZE`.
- Applies ImageNet-standard normalization (because the CNN backbone uses ImageNet pretrained weights - stated explicitly).
- Returns `(image_tensor, mos_score)` where `mos_score = NaN` if no label exists.


In [ ]:
# ImageNet normalisation - required because CNN backbone uses ImageNet pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(img_size: int, train: bool = True, extra_aug: bool = False) -> T.Compose:
    '''Returns torchvision transform pipeline.'''
    tfms = []
    tfms.append(T.Resize((img_size, img_size)))
    if train:
        tfms += [
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            # v3b: 90-degree rotations are label-preserving (no resampling on a square image)
            *([T.RandomChoice([T.Lambda(lambda im, a=a: im.rotate(a)) for a in (0, 90, 180, 270)])]
              if extra_aug else []),
            # Mild photometric jitter only: saturation/hue jitter would alter colour cast,
            # a key underwater quality cue, and therefore the label itself.
            T.ColorJitter(brightness=0.1, contrast=0.1),
            # Square crops with <=10% zoom, to limit resampling blur
            T.RandomResizedCrop(img_size, scale=(0.9, 1.0), ratio=(1.0, 1.0)),
        ]
    tfms += [
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return T.Compose(tfms)


class UWIQADataset(Dataset):
    '''
    Underwater Image Quality Assessment dataset.
    Returns (image_tensor, mos_float) per image.
    mos_float = float("nan") when no label is available.
    '''
    def __init__(self, df: pd.DataFrame, img_size: int, train: bool = False, extra_aug: bool = False):
        self.df = df.reset_index(drop=True)
        self.transform = get_transforms(img_size, train=train, extra_aug=extra_aug)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        img_tensor = self.transform(img)                          # (3, H, W)
        mos = float(row["mos"]) if not pd.isna(row["mos"]) else float("nan")
        return img_tensor, torch.tensor(mos, dtype=torch.float32)


def build_loaders(df: pd.DataFrame, cfg: dict) -> Dict[str, DataLoader]:
    from torch.utils.data import RandomSampler, WeightedRandomSampler
    loaders = {}
    for split in ["train", "val", "test"]:
        sub = df[df["split"] == split]
        if len(sub) == 0:
            continue
        ds = UWIQADataset(sub, cfg["IMAGE_SIZE"], train=(split=="train"),
                         extra_aug=(split=="train" and cfg.get("BALANCE_CLASSES", False) and cfg.get("BALANCE_ROT90", False)))
        if split == "train":
            # Iterations per epoch = CFG["BATCHES_PER_EPOCH"]
            n_draw = cfg.get("BATCHES_PER_EPOCH", 25) * cfg["BATCH_SIZE"]
            if cfg.get("BALANCE_CLASSES", False) and sub["mos"].notna().all():
                # Class-balanced sampling (train only). Each draw is augmented independently.
                w = class_balance_weights(sub["mos"].values, cfg)
                sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                                num_samples=n_draw, replacement=True)
            else:
                sampler = RandomSampler(ds, replacement=True, num_samples=n_draw)
            loaders[split] = DataLoader(
                ds,
                batch_size=cfg["BATCH_SIZE"],
                sampler=sampler,
                num_workers=0,
                pin_memory=(DEVICE.type=="cuda"),
            )
        else:
            loaders[split] = DataLoader(
                ds,
                batch_size=cfg["BATCH_SIZE"],
                shuffle=False,
                num_workers=0,
                pin_memory=(DEVICE.type=="cuda"),
                drop_last=False,
            )
    return loaders


loaders = build_loaders(df_all, CFG)
for s, ld in loaders.items():
    print(f"  {s:5s}: {len(ld.dataset)} images, {len(ld)} batches")

# Quick shape check
_imgs, _mos = next(iter(loaders["train"]))
print(f"\nSample batch - image: {_imgs.shape}  mos: {_mos.shape}")


## 12 . CNN Patch Feature Extractor

**Why CNN before Transformer?**

- **CNN** excels at extracting *local* visual features - edges, textures, colour patterns - within a small receptive field. This is ideal for characterizing an individual patch's visual appearance.
- **Transformer** excels at modelling *long-range dependencies* across the entire token sequence. It requires meaningful token representations as input, which the CNN provides.

Using CNN -> Transformer follows the same motivation as ViT/DeiT: CNN as a perceptual encoder, Transformer as a relational reasoner.

**Implementation decision:** We project the CNN output to `EMBED_DIM` using a linear layer so all scales share the same Transformer embedding space. The CNN backbone is **ImageNet-pretrained** (stated explicitly). ImageNet preprocessing (mean/std normalization) is applied in the dataset class.


In [ ]:
class SEBlock(nn.Module):
    '''
    Squeeze-and-Excitation channel attention.
    Gate = 2*sigmoid(z) with the last layer zero-initialised, so the block starts
    as an exact identity and does not disturb the pretrained ImageNet features.
    '''
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = x.mean(dim=(2, 3))                                    # squeeze (M, C)
        g = 2.0 * torch.sigmoid(self.fc2(F.relu(self.fc1(s))))    # excite  (M, C)
        return x * g.unsqueeze(-1).unsqueeze(-1)


class CNNPatchEncoder(nn.Module):
    '''
    Extracts a d-dimensional feature vector from each patch.

    Input : (B*N, 3, p, p)   Output: (B*N, embed_dim)

    Backbones
      resnet18 / resnet50    : ImageNet CNN, SE block, global average pool.
      dinov2_vits14/vitb14   : DINOv2 ViT via timm. Each patch is bilinearly resized to
                               input_size x input_size first (a ViT needs a fixed grid);
                               the CLS token is the patch feature.
      swin_t / swin_s        : ImageNet-pretrained Swin Transformer via timm. Used as a WHOLE-IMAGE
                               dense encoder (forward_grid): the image is resized to 224, one pass
                               gives a 7x7 feature map, which is resized to the finest token grid
                               and average-pooled for the coarser scales, so the token layout (e.g.
                               64 + 16 + 4 = 84) is identical to the patch path. Running Swin on every
                               patch at 224 would cost ~64x more compute and not fit in memory.
                               forward() on a patch tensor still works (resize + pooled feature).
    All paths expose self.features (the backbone), so freezing and the backbone LR
    multiplier behave identically for any of them.
    '''
    DINO_TIMM = {
        "dinov2_vits14": "vit_small_patch14_dinov2.lvd142m",
        "dinov2_vitb14": "vit_base_patch14_dinov2.lvd142m",
    }

    # Pretrained Swin weights in timm: ImageNet-22k pretraining fine-tuned on 1k first, plain 1k as fallback
    SWIN_TIMM = {
        "swin_t": ("swin_tiny_patch4_window7_224.ms_in22k_ft_in1k",  "swin_tiny_patch4_window7_224.ms_in1k"),
        "swin_s": ("swin_small_patch4_window7_224.ms_in22k_ft_in1k", "swin_small_patch4_window7_224.ms_in1k"),
    }

    def __init__(self, backbone_name: str, embed_dim: int, pretrained: bool = True,
                 use_se: bool = True, input_size: Optional[int] = None):
        super().__init__()
        self.embed_dim = embed_dim
        self.is_swin = backbone_name in self.SWIN_TIMM
        self.is_dense = self.is_swin                      # Swin runs on the whole image (forward_grid)
        self.is_vit = backbone_name in self.DINO_TIMM or self.is_swin

        if self.is_swin:
            import timm
            self.input_size = int(input_size or 224)
            assert self.input_size == 224, "CNN_INPUT_SIZE must be 224 for the pretrained Swin backbones"
            last_err = None
            for tag in self.SWIN_TIMM[backbone_name]:
                try:
                    self.features = timm.create_model(tag, pretrained=pretrained, num_classes=0)
                    print(f"  timm weights: {tag}")
                    break
                except Exception as e:                    # unknown tag or download failure: try the next one
                    last_err = e
                    print(f"  could not load {tag}: {type(e).__name__}: {e}")
            else:
                raise RuntimeError(f"No Swin weights could be loaded for '{backbone_name}'") from last_err
            cnn_out_dim = self.features.num_features
            self.se = nn.Identity()
            self.pool = nn.Identity()
        elif self.is_vit:
            import timm
            self.input_size = int(input_size or 224)
            assert self.input_size % 14 == 0, "CNN_INPUT_SIZE must be a multiple of 14 for DINOv2"
            self.features = timm.create_model(
                self.DINO_TIMM[backbone_name], pretrained=pretrained,
                num_classes=0, img_size=self.input_size,
            )
            cnn_out_dim = self.features.num_features
            self.se = nn.Identity()
            self.pool = nn.Identity()
        else:
            self.input_size = None
            weights = "IMAGENET1K_V1" if pretrained else None
            if backbone_name == "resnet18":
                base = tvm.resnet18(weights=weights)
                cnn_out_dim = 512
            elif backbone_name == "resnet50":
                base = tvm.resnet50(weights=weights)
                cnn_out_dim = 2048
            else:
                raise ValueError(f"Unsupported backbone: {backbone_name}")
            # Remove avgpool and fc; keep feature extractor up to layer4
            self.features = nn.Sequential(*list(base.children())[:-2])
            self.se = SEBlock(cnn_out_dim) if use_se else nn.Identity()
            self.pool = nn.AdaptiveAvgPool2d(1)

        self.cnn_out_dim = cnn_out_dim
        self.proj = nn.Linear(cnn_out_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        state = "ImageNet weights" if pretrained else "random init"
        print(f"  backbone '{backbone_name}' loaded ({state}), feature dim {cnn_out_dim}.")

    def forward_grid(self, images: torch.Tensor, patch_sizes: List[int]) -> Dict[int, torch.Tensor]:
        '''
        Swin only. images (B, 3, H, W) -> {patch_size: (B, (H//p)**2, embed_dim)}.
        One backbone pass per image; the final feature map is resized to the finest token grid
        (H // min(patch_sizes)) and average-pooled for each coarser scale.
        '''
        B, _, H, W = images.shape
        x = images
        if H != self.input_size or W != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        f = self.features.forward_features(x)
        if f.dim() == 3:                                   # (B, L, C) in older timm
            n = int(round(f.shape[1] ** 0.5))
            f = f.reshape(B, n, n, -1)
        if f.shape[-1] != self.cnn_out_dim:                # NCHW -> NHWC
            f = f.permute(0, 2, 3, 1)
        f = f.permute(0, 3, 1, 2).contiguous()             # (B, C, h, w)
        g0 = H // min(patch_sizes)
        base = F.interpolate(f, size=(g0, g0), mode="bilinear", align_corners=False)
        out: Dict[int, torch.Tensor] = {}
        for p in patch_sizes:
            g = H // p
            grid = base if g == g0 else F.adaptive_avg_pool2d(base, g)
            tok = grid.flatten(2).transpose(1, 2)           # (B, g*g, C), row-major like the patch grid
            out[p] = self.norm(self.proj(tok))              # (B, g*g, embed_dim)
        return out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_vit:
            if x.shape[-1] != self.input_size:
                x = F.interpolate(x, size=(self.input_size, self.input_size),
                                  mode="bilinear", align_corners=False)
            feat = self.features(x)                  # (M, C) CLS token
        else:
            feat = self.features(x)                  # (M, C_cnn, h', w')
            feat = self.se(feat)                     # channel attention (identity at init)
            feat = self.pool(feat).flatten(1)        # (M, C_cnn)
        feat = self.proj(feat)                       # (M, embed_dim)
        feat = self.norm(feat)                       # (M, embed_dim)
        return feat


# -- Sanity check -------------------------------------------------------------
_enc = CNNPatchEncoder(CFG["CNN_BACKBONE"], CFG["EMBED_DIM"], CFG["CNN_PRETRAINED"],
                       input_size=CFG["CNN_INPUT_SIZE"])
for p in CFG["PATCH_SIZES"]:
    N = (CFG["IMAGE_SIZE"]//p)**2
    _dummy_patches = torch.randn(2*N, 3, p, p)
    _out = _enc(_dummy_patches)
    assert _out.shape == (2*N, CFG["EMBED_DIM"]), f"CNN output shape error for p={p}"
if getattr(_enc, "is_dense", False):
    _g = _enc.forward_grid(torch.randn(2, 3, CFG["IMAGE_SIZE"], CFG["IMAGE_SIZE"]), CFG["PATCH_SIZES"])
    for p in CFG["PATCH_SIZES"]:
        assert _g[p].shape == (2, (CFG["IMAGE_SIZE"] // p) ** 2, CFG["EMBED_DIM"]), f"Swin grid shape error for p={p}"
    print("OK Swin forward_grid token shapes match the patch layout.")
print("OK CNN encoder output shape checks passed for all patch sizes.")
del _enc


## 13 . Transformer Cross-Patch Context Encoder

### Role

After CNN encoding, each patch is a vector $\mathbf{f}_i \in \mathbb{R}^d$. The Transformer allows every patch to attend to every other patch:

$$\mathbf{A} = \text{Softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right), \quad \mathbf{X}' = \mathbf{A}\mathbf{V}$$

This means: *a patch that belongs to a fish subject will "know" about the surrounding water background*, and vice versa. This contextual information is critical for computing perceptual importance ($\alpha_i$) correctly.

### Scale Embeddings

Because all three scales are concatenated into a single token sequence, the Transformer must be able to distinguish which scale a token belongs to. We add a **learnable scale embedding** $\mathbf{e}_s \in \mathbb{R}^d$ to each token before the Transformer:

$$\text{token}_i = \mathbf{f}_i^{(s)} + \mathbf{e}_s$$

This is analogous to positional embeddings in BERT/ViT.


In [ ]:
class TransformerEncoderBlock(nn.Module):
    '''
    Single Transformer encoder block:
      LayerNorm -> Multi-Head Self-Attention -> Residual
      LayerNorm -> Feed-Forward (GELU) -> Residual

    Implements equations (66-68) from the design document:
      Q=XW_Q, K=XW_K, V=XW_V
      A = Softmax(QK^T / sqrt(d))
      X_prime = AV   (then residual + FFN)
    '''
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff    = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Args:
            x : (B, N, d)
        Returns:
            (B, N, d)
        '''
        # Self-attention with pre-norm
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)   # Q=K=V
        x = x + attn_out                                    # residual

        # Feed-forward with pre-norm
        x = x + self.ff(self.norm2(x))                     # residual
        return x


class PatchTransformerEncoder(nn.Module):
    '''
    Full Transformer encoder for all multi-scale patch tokens.

    Adds:
      - Scale embeddings  (one per scale)
      - Positional embeddings (one per token position within a scale)
      - L stacked TransformerEncoderBlock layers

    Input : per-scale feature dicts
            {patch_size: (B, N_s, embed_dim)}
    Output: contextualized features (B, N_total, embed_dim)
            where N_total = sum of N_s across all scales
    '''
    def __init__(
        self,
        patch_sizes: List[int],
        img_size: int,
        embed_dim: int,
        depth: int,
        num_heads: int,
        ff_dim: int,
        dropout: float,
    ):
        super().__init__()
        self.patch_sizes = patch_sizes
        self.embed_dim   = embed_dim

        # Learnable scale embeddings (one per scale)
        self.scale_embed = nn.Embedding(len(patch_sizes), embed_dim)

        # Learnable positional embeddings (one per token per scale)
        self.pos_embeds = nn.ParameterDict()
        for p in patch_sizes:
            N = (img_size // p) ** 2
            self.pos_embeds[str(p)] = nn.Parameter(torch.randn(1, N, embed_dim) * 0.02)

        # Stacked Transformer blocks
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(
        self,
        scale_features: Dict[int, torch.Tensor],
        film: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, Dict[int, Tuple[int,int]]]:
        '''
        Args:
            scale_features: {patch_size: (B, N_s, embed_dim)}

        Returns:
            h      : (B, N_total, embed_dim)  contextualized tokens
            slices : {patch_size: (start_idx, end_idx)} for reconstructing scales
        '''
        tokens_list = []
        slices: Dict[int, Tuple[int,int]] = {}
        cursor = 0

        for s_idx, p in enumerate(self.patch_sizes):
            feats = scale_features[p]                           # (B, N_s, d)
            N_s   = feats.shape[1]
            # Add scale embedding (same for all tokens at this scale)
            feats = feats + self.scale_embed.weight[s_idx].unsqueeze(0).unsqueeze(0)
            # Add positional embedding
            feats = feats + self.pos_embeds[str(p)]
            tokens_list.append(feats)
            slices[p] = (cursor, cursor + N_s)
            cursor += N_s

        x = torch.cat(tokens_list, dim=1)                      # (B, N_total, d)
        if film is not None:
            gamma, beta = film                    # each (B, 1, d): FiLM from degradation embedding
            x = x * (1.0 + gamma) + beta

        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)

        return x, slices


## 14 . Three-Branch Head (Solution II integrated into Solution III)

After the Transformer produces contextualized patch representations $h_i$, **three parallel branches** are applied (Solution II), as required by the integration specification.

### Quality Branch
$$q_i = \sigma(W_q h_i + b_q), \quad q_i \in [0,1]$$

### Saliency Branch  
$$s_i = \sigma(W_s h_i + b_s), \quad s_i \in [0,1]$$

### Degradation Branch
$$\mathbf{r}_i = [r_i^{\text{color}}, r_i^{\text{haze}}, r_i^{\text{blur}}, r_i^{\text{contrast}}, r_i^{\text{noise}}]$$
$$d_i = \sum_k \beta_k r_i^{(k)}, \quad \sum_k \beta_k = 1$$

**Important distinction:**
1. **Ground-truth degradation labels** - per-patch labels from the dataset (rare; declared via `HAS_DEGRADATION_GT`).  
2. **Pseudo-labels** - generated from synthetic degradation pipelines as a weak supervisory signal.  
3. **Latent predictions** - $\hat{\mathbf{r}}_i$ learned only through image-level MOS supervision (the default in this implementation).  

This notebook defaults to case 3. The degradation loss $\mathcal{L}_\text{deg}$ is **automatically disabled** unless `HAS_DEGRADATION_GT = True`.


In [ ]:
class ThreeBranchHead(nn.Module):
    '''
    Applied to each contextualized patch token h_i from the Transformer.

    Branches (from Solution II, applied post-Transformer per integration requirement):
      Quality     : q_i = sigma(Wq h_i + bq)
      Saliency    : s_i = sigma(Ws h_i + bs)
      Degradation : r_i = [r_color, r_haze, r_blur, r_contrast, r_noise]
                    d_i = softmax(beta)^T r_i

    Input : (B, N, embed_dim)
    Output: q (B,N,1), s (B,N,1), r (B,N,K), d (B,N,1)

    Degradation note:
      r_i values are in [0,1] (sigmoid activated).
      d_i is a weighted sum (interpretable degradation severity).
      These are LATENT PREDICTIONS learned via image-level MOS.
      They are NOT ground-truth degradation unless a labelled dataset is used.
    '''
    def __init__(self, embed_dim: int, n_deg_attrs: int = 5):
        super().__init__()
        self.n_deg_attrs = n_deg_attrs

        # Quality branch: single linear + sigmoid
        self.quality_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Linear(embed_dim // 2, 1),
        )

        # Saliency branch: single linear + sigmoid
        self.saliency_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Linear(embed_dim // 2, 1),
        )

        # Degradation branch: predict K attributes
        self.degradation_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Linear(embed_dim // 2, n_deg_attrs),
        )

        # Learnable combination weights beta_k (constrained to sum to 1 via softmax)
        self.deg_weights = nn.Parameter(torch.ones(n_deg_attrs) / n_deg_attrs)

    def forward(
        self, h: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        '''
        Args:
            h : (B, N, embed_dim)
        Returns:
            q : (B, N, 1)  patch quality scores in [0,1]
            s : (B, N, 1)  saliency scores in [0,1]
            r : (B, N, K)  per-attribute degradation scores in [0,1]
            d : (B, N, 1)  combined degradation severity in [0,1]
        '''
        q = torch.sigmoid(self.quality_head(h))          # (B, N, 1)
        s = torch.sigmoid(self.saliency_head(h))         # (B, N, 1)
        r = torch.sigmoid(self.degradation_head(h))      # (B, N, K)

        beta = F.softmax(self.deg_weights, dim=0)        # (K,)  sums to 1
        d = (r * beta.unsqueeze(0).unsqueeze(0)).sum(-1, keepdim=True)  # (B,N,1)

        return q, s, r, d


## 15 . Importance Fusion and Hierarchical Scale-Adaptive Pooling

### Importance Fusion

For each patch, we combine the contextualized Transformer feature $h_i$, quality $q_i$, saliency $s_i$, and degradation $d_i$ into a scalar importance energy:

$$e_i = G(h_i, q_i, s_i, d_i)$$

$$\alpha_i = \frac{\exp(e_i)}{\sum_j \exp(e_j)}$$

The function $G$ is implemented as a small MLP applied to the concatenation $[h_i; q_i; s_i; d_i]$.

### Scale-Adaptive Pooling

For each scale $s$:
$$Q^{(s)} = \sum_i \alpha_i^{(s)} q_i^{(s)}$$

The scale-level quality scores are combined with **learnable scale weights** $\lambda_s$ (constrained to sum to 1 via softmax):
$$\hat{Q} = \sum_s \lambda_s Q^{(s)}, \quad \sum_s \lambda_s = 1$$

The scale weights are data-driven: the model learns which patch resolution is most informative for quality prediction for this particular dataset.


In [ ]:
class ImportanceFusionPooling(nn.Module):
    '''
    Combines q_i, s_i, d_i, h_i into importance weights alpha_i,
    then performs scale-adaptive weighted pooling.

    Equations:
      e_i = MLP([h_i; q_i; s_i; d_i])
      alpha_i = softmax(e_i)  over N patches in a scale
      Q^(s) = Sigma_i alpha_i^(s) q_i^(s)
      Q_hat = Sigma_s lambda_s Q^(s)   where lambda_s = softmax(scale_logits)

    Input:
      h (B, N, d), q (B, N, 1), s (B, N, 1), d (B, N, 1)
      slices: dict of {patch_size: (start, end)} into the N dimension
    Output:
      Q_hat   : (B,)  final image quality score
      alphas  : (B, N, 1)  importance weights
      lambdas : (S,)  scale weights
    '''
    def __init__(self, embed_dim: int, n_scales: int):
        super().__init__()
        # Importance MLP: input = [h, q, s, d] = embed_dim + 3
        in_dim = embed_dim + 3
        self.importance_mlp = nn.Sequential(
            nn.Linear(in_dim, embed_dim // 2),
            nn.GELU(),
            nn.Linear(embed_dim // 2, 1),
        )
        # Learnable scale weights (not from document; implementation decision)
        self.scale_logits = nn.Parameter(torch.zeros(n_scales))
        # Per-image scale attention: zero-init, so it starts identical to the global weights
        self.scale_attn = nn.Linear(embed_dim, 1)
        nn.init.zeros_(self.scale_attn.weight)
        nn.init.zeros_(self.scale_attn.bias)

    def forward(
        self,
        h: torch.Tensor,
        q: torch.Tensor,
        s: torch.Tensor,
        d: torch.Tensor,
        slices: Dict[int, Tuple[int,int]],
        patch_sizes: List[int],
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        '''
        Args:
            h : (B, N_total, d)
            q : (B, N_total, 1)
            s : (B, N_total, 1)
            d : (B, N_total, 1)
        Returns:
            Q_hat   : (B,)
            alphas  : (B, N_total, 1)
            lambdas : (n_scales,)
        '''
        # -- Compute per-patch importance energies ----------------------
        inp   = torch.cat([h, q, s, d], dim=-1)     # (B, N, d+3)
        e     = self.importance_mlp(inp)              # (B, N, 1)

        # -- Per-scale softmax normalization ----------------------------
        scale_qualities = []
        scale_ctx = []
        alphas_full = torch.zeros_like(e)

        for p in patch_sizes:
            start, end = slices[p]
            e_s = e[:, start:end, :]                  # (B, N_s, 1)
            alpha_s = F.softmax(e_s, dim=1)           # (B, N_s, 1) sums to 1 over N_s
            alphas_full[:, start:end, :] = alpha_s

            q_s = q[:, start:end, :]                  # (B, N_s, 1)
            Q_s = (alpha_s * q_s).sum(dim=1)          # (B, 1)
            scale_qualities.append(Q_s)
            scale_ctx.append(self.scale_attn(h[:, start:end, :].mean(dim=1)))   # (B, 1)

        # -- Scale-adaptive fusion --------------------------------------
        ctx       = torch.cat(scale_ctx, dim=-1)                                   # (B, S)
        lambdas_b = F.softmax(self.scale_logits.unsqueeze(0) + ctx, dim=-1)        # (B, S) per image
        Q_stack   = torch.cat(scale_qualities, dim=-1)                             # (B, S)
        Q_hat     = (Q_stack * lambdas_b).sum(-1)                                  # (B,)
        lambdas   = lambdas_b.mean(dim=0)    # (S,) batch-mean scale weights, sums to 1 (for logging)

        return Q_hat, alphas_full, lambdas


## 15b · Degradation Estimator + FiLM feedback (v3)

**Stage 1 (label-free w.r.t. MOS):** `DegradationEstimator` is pretrained to regress the known levels of synthetic haze, colour cast and blur applied to pool images. Cached to `checkpoints/deg_pretrained_emb*.pt`.

**Stage 2:** the estimator's embedding `z` drives a zero-initialised `FiLM` layer that rescales and shifts every Transformer token, so the degradation estimate conditions the context modelling. The estimator is then fine-tuned with the rest of the model.

> Note: EDANet's DIDNet is pretrained against a pseudo-reference residual. UWIQA has no clean reference images, so this uses synthetic levels instead. It is a different pretext task and has not been validated.


In [ ]:
from torch.utils.data import RandomSampler
import math
import torchvision.transforms.functional as TF
from torch.optim.lr_scheduler import CosineAnnealingLR


class DegradationEstimator(nn.Module):
    '''
    Image-level degradation embedding z (DEG_EMBED_DIM) plus 3 interpretable levels
    in [0,1]: haze, colour cast, blur. Input is ImageNet-normalised (B, 3, H, W).
    '''
    N_LEVELS = 3

    def __init__(self, emb_dim: int = 128, norm: str = None):
        super().__init__()
        norm = norm or CFG.get("DEG_NORM", "group")
        def blk(i, o, s):
            # Trained from scratch, so GroupNorm (no dependence on batch statistics) by default
            nrm = nn.GroupNorm(8, o) if norm == "group" else nn.BatchNorm2d(o)
            return nn.Sequential(nn.Conv2d(i, o, 3, s, 1, bias=False), nrm, nn.ReLU(inplace=True))
        self.body = nn.Sequential(blk(3, 32, 2), blk(32, 64, 2), blk(64, 128, 2), blk(128, 128, 2),
                                  nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.emb = nn.Sequential(nn.Linear(128, emb_dim), nn.GELU(), nn.LayerNorm(emb_dim))
        self.level_head = nn.Linear(emb_dim, self.N_LEVELS)

    def forward(self, x: torch.Tensor):
        z = self.emb(self.body(x))                      # (B, emb_dim)
        levels = torch.sigmoid(self.level_head(z))      # (B, 3)
        return z, levels


class FiLM(nn.Module):
    '''
    Feature-wise linear modulation of Transformer tokens from the degradation embedding.
    Zero-initialised: gamma = beta = 0 at start, so tokens are unchanged (v2 behaviour).
    '''
    def __init__(self, cond_dim: int, embed_dim: int):
        super().__init__()
        self.gb = nn.Linear(cond_dim, 2 * embed_dim)
        nn.init.zeros_(self.gb.weight)
        nn.init.zeros_(self.gb.bias)

    def forward(self, z: torch.Tensor):
        gamma, beta = self.gb(z).chunk(2, dim=-1)       # (B, d) each
        return gamma.unsqueeze(1), beta.unsqueeze(1)    # (B, 1, d): broadcast over tokens


# Synthetic degradation model. Levels are sampled uniformly in [0,1] and applied to
# training images, so the targets are known exactly.
_AIRLIGHT = (0.20, 0.50, 0.60)    # bluish-green veil of turbid water (illustrative values)
_CAST_DIR = (-0.5, -0.1, 0.15)    # red attenuated most, blue least (water absorption order)


def apply_synthetic_degradation(raw: torch.Tensor):
    '''
    raw : (B, 3, H, W) in [0,1].
    Returns (degraded raw images, levels (B, 3)) where levels = [haze, cast, blur] in [0,1].
    '''
    B, dev = raw.shape[0], raw.device
    lev = torch.rand(B, 3, device=dev)
    air = torch.tensor(_AIRLIGHT, device=dev).view(1, 3, 1, 1)
    cdir = torch.tensor(_CAST_DIR, device=dev).view(1, 3, 1, 1)
    out = []
    for i in range(B):
        x = raw[i:i + 1]
        x = (x * (1.0 + 0.6 * lev[i, 1] * cdir)).clamp(0, 1)      # colour cast
        t = 0.6 * lev[i, 0]
        x = x * (1 - t) + air * t                                  # haze (veil toward airlight)
        sigma = 2.5 * float(lev[i, 2])                             # blur, up to 2.5 px
        if sigma > 0.1:
            k = 2 * int(math.ceil(3 * sigma)) + 1
            x = TF.gaussian_blur(x, kernel_size=[k, k], sigma=[sigma, sigma])
        out.append(x)
    return torch.cat(out, dim=0), lev


def deg_ckpt_path(cfg, tag: str = None) -> Path:
    '''Per-fold checkpoint when tag is given (e.g. "s42_f3"), so each fold pretrains on its own train images only.'''
    sfx = "_gn" if cfg.get("DEG_NORM", "group") == "group" else ""   # GN and BN weights are not interchangeable
    t = f"_{tag}" if tag else ""
    return Path(cfg["CHECKPOINT_DIR"]) / f"deg_pretrained_emb{cfg['DEG_EMBED_DIM']}{sfx}{t}.pt"


def pretrain_degradation_estimator(est: nn.Module, pool_df: pd.DataFrame, cfg: dict, ckpt_path: Path):
    '''
    Stage 1 (label-free w.r.t. MOS): regress the known synthetic levels from the degraded image.
    Uses only the images in pool_df (the fold's training images; never validation or test). Cached to ckpt_path with an image-set signature.
    '''
    import hashlib
    # Provenance: a cached checkpoint is reused only if it was trained on exactly these images
    sig = hashlib.md5("|".join(sorted(map(str, pool_df["image_name"]))).encode()).hexdigest()
    if Path(ckpt_path).exists():
        ck = torch.load(ckpt_path, map_location=DEVICE)
        if ck.get("images_sig") == sig:
            est.load_state_dict(ck["state_dict"])
            print(f"Loaded degradation estimator from {ckpt_path} (image set matches)")
            return est
        print(f"Cached {Path(ckpt_path).name} was trained on a different image set; retraining.")

    ds = UWIQADataset(pool_df, cfg["IMAGE_SIZE"], train=False)
    sampler = RandomSampler(ds, replacement=True, num_samples=cfg["DEG_PRETRAIN_ITERS"] * cfg["BATCH_SIZE"])
    ld = DataLoader(ds, batch_size=cfg["BATCH_SIZE"], sampler=sampler, num_workers=0, drop_last=True)
    opt = torch.optim.AdamW(est.parameters(), lr=cfg["DEG_PRETRAIN_LR"], weight_decay=cfg["WEIGHT_DECAY"])
    sched = CosineAnnealingLR(opt, T_max=cfg["DEG_PRETRAIN_EPOCHS"])
    mean = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)

    print(f"Degradation pretraining on {len(pool_df)} pool images (synthetic levels, no MOS used).")
    for ep in range(1, cfg["DEG_PRETRAIN_EPOCHS"] + 1):
        est.train()
        tot, n = 0.0, 0
        for x_norm, _ in ld:
            x_norm = x_norm.to(DEVICE)
            with torch.no_grad():
                raw = (x_norm * std + mean).clamp(0, 1)
                deg_raw, lev = apply_synthetic_degradation(raw)
                deg_norm = (deg_raw - mean) / std
            _, pred = est(deg_norm)
            loss = F.mse_loss(pred, lev)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item()
            n += 1
        sched.step()
        print(f"  [deg-pretrain] epoch {ep:02d}/{cfg['DEG_PRETRAIN_EPOCHS']}  level MSE={tot / max(n, 1):.4f}")
    torch.save({"state_dict": est.state_dict(), "images_sig": sig, "n_images": len(pool_df)}, ckpt_path)
    print(f"Saved degradation estimator to {ckpt_path}")
    return est


## 16 . Full Hybrid Model

This assembles all components into the complete pipeline described in the design document Section11 (equation 120):

```
I -> Multi-Scale Patch Extraction
  -> CNN Feature Extraction
  -> [Scale Embedding + Position Embedding]
  -> Transformer Context Modeling
  -> Quality / Saliency / Degradation Branches
  -> Importance Fusion
  -> Hierarchical Scale-Adaptive Pooling
  -> Global UIQA Score Q
```


In [ ]:
class HybridUIQAModel(nn.Module):
    '''
    Full hybrid UIQA model (Solution III backbone + Solution II head).

    v3 additions (switchable via CFG):
      USE_FILM : DegradationEstimator predicts an image-level embedding z from the image;
                 FiLM(z) rescales and shifts every Transformer token. Zero-initialised,
                 so the model starts identical to v2.
    '''
    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg         = cfg
        self.patch_sizes = cfg["PATCH_SIZES"]
        self.embed_dim   = cfg["EMBED_DIM"]
        self.use_film    = bool(cfg.get("USE_FILM", False))

        # -- CNN / ViT backbone (shared across all scales) --------------
        self.cnn = CNNPatchEncoder(
            cfg["CNN_BACKBONE"], cfg["EMBED_DIM"], cfg["CNN_PRETRAINED"],
            use_se=cfg.get("USE_SE", True), input_size=cfg.get("CNN_INPUT_SIZE"),
        )

        # -- Transformer ------------------------------------------------
        self.transformer = PatchTransformerEncoder(
            patch_sizes  = cfg["PATCH_SIZES"],
            img_size     = cfg["IMAGE_SIZE"],
            embed_dim    = cfg["EMBED_DIM"],
            depth        = cfg["TRANSFORMER_DEPTH"],
            num_heads    = cfg["TRANSFORMER_HEADS"],
            ff_dim       = cfg["TRANSFORMER_FF_DIM"],
            dropout      = cfg["TRANSFORMER_DROPOUT"],
        )

        # -- Three-branch head (Solution II) ----------------------------
        self.head = ThreeBranchHead(cfg["EMBED_DIM"], cfg["N_DEGRADATION_ATTRS"])

        # -- Importance fusion + pooling --------------------------------
        self.pooling = ImportanceFusionPooling(
            embed_dim = cfg["EMBED_DIM"],
            n_scales  = len(cfg["PATCH_SIZES"]),
        )

        # -- Degradation estimator + FiLM feedback (v3) -----------------
        if self.use_film:
            self.deg_est = DegradationEstimator(cfg["DEG_EMBED_DIM"])
            self.film    = FiLM(cfg["DEG_EMBED_DIM"], cfg["EMBED_DIM"])

    def forward(self, images: torch.Tensor) -> Dict[str, torch.Tensor]:
        '''
        Args:
            images : (B, 3, H, W)  ImageNet-normalized

        Returns dict with keys:
            Q_hat, q, s, r, d, alphas, lambdas, slices, deg_levels (None if USE_FILM is off)
        '''
        B = images.shape[0]

        # -- 1. Multi-scale patch extraction ---------------------------
        # -- 2. CNN / ViT feature extraction per scale -----------------
        scale_features: Dict[int, torch.Tensor] = {}
        if getattr(self.cnn, "is_dense", False):
            # Swin: one whole-image pass, token grids pooled to the same layout as the patch grids
            scale_features = self.cnn.forward_grid(images, self.patch_sizes)
        else:
            patches = extract_patches_multiscale(images, self.patch_sizes)
            for p in self.patch_sizes:
                pts = patches[p]                              # (B, N_s, 3, p, p)
                N_s = pts.shape[1]
                feats = self.cnn(pts.reshape(B * N_s, 3, p, p))
                scale_features[p] = feats.reshape(B, N_s, self.embed_dim)

        # -- 2b. Degradation embedding -> FiLM (v3) ---------------------
        film, deg_levels = None, None
        if self.use_film:
            z, deg_levels = self.deg_est(images)
            film = self.film(z)

        # -- 3. Transformer context modeling ---------------------------
        h, slices = self.transformer(scale_features, film=film)

        # -- 4. Three-branch head ---------------------------------------
        q, s, r, d = self.head(h)

        # -- 5. Importance fusion + pooling ----------------------------
        Q_hat, alphas, lambdas = self.pooling(h, q, s, d, slices, self.patch_sizes)

        return {
            "Q_hat"      : Q_hat,
            "q"          : q,
            "s"          : s,
            "r"          : r,
            "d"          : d,
            "alphas"     : alphas,
            "lambdas"    : lambdas,
            "slices"     : slices,
            "deg_levels" : deg_levels,
        }


# -- Instantiate ---------------------------------------------------------------
model = HybridUIQAModel(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model instantiated.  Trainable parameters: {n_params:,}")


## 17 . Mandatory Sanity Checks

All checks run before training. A failed check raises a clear error.

In [ ]:
def run_sanity_checks(model: nn.Module, cfg: dict) -> None:
    '''Run all mandatory checks before training.'''
    model.eval()
    H = W = cfg["IMAGE_SIZE"]
    B = 2
    dummy = torch.randn(B, 3, H, W).to(DEVICE)

    print("Running sanity checks...")
    with torch.no_grad():
        out = model(dummy)

    Q_hat   = out["Q_hat"]
    q       = out["q"]
    s       = out["s"]
    r       = out["r"]
    d       = out["d"]
    alphas  = out["alphas"]
    lambdas = out["lambdas"]

    N_total = sum((H//p)**2 for p in cfg["PATCH_SIZES"])
    K = cfg["N_DEGRADATION_ATTRS"]

    # 1. Image tensor shape
    assert dummy.shape == (B, 3, H, W),           "ERROR Image tensor shape"
    print(f"  v Image tensor       : {tuple(dummy.shape)}")

    # 2. Patch counts
    for p in cfg["PATCH_SIZES"]:
        N_s = (H//p)**2
        start, end = out["slices"][p]
        assert (end - start) == N_s,              f"ERROR Patch count for p={p}"
    print(f"  v Patch counts       : {[((H//p)**2) for p in cfg['PATCH_SIZES']]} (total {N_total})")

    # 3. Transformer output / token count
    assert q.shape == (B, N_total, 1),            f"ERROR q shape: {q.shape}"
    print(f"  v Total tokens       : {N_total}")

    # 4. Quality output range
    assert (q >= 0).all() and (q <= 1).all(),     "ERROR q not in [0,1]"
    print(f"  v Quality range      : [{q.min():.3f}, {q.max():.3f}] <= [0,1]")

    # 5. Saliency output range
    assert (s >= 0).all() and (s <= 1).all(),     "ERROR s not in [0,1]"
    print(f"  v Saliency range     : [{s.min():.3f}, {s.max():.3f}] <= [0,1]")

    # 6. Degradation output range
    assert (r >= 0).all() and (r <= 1).all(),     "ERROR r not in [0,1]"
    print(f"  v Degradation range  : [{r.min():.3f}, {r.max():.3f}] <= [0,1]")

    # 7. Importance weights sum to 1 per scale
    for p in cfg["PATCH_SIZES"]:
        start, end = out["slices"][p]
        alpha_s = alphas[:, start:end, :]
        sums = alpha_s.sum(dim=1)       # (B, 1)
        assert torch.allclose(sums, torch.ones_like(sums), atol=1e-4), \
            f"ERROR Importance weights do not sum to 1 for scale {p}"
    print(f"  v Importance weights : sum to 1 per scale v")

    # 8. Scale weights sum to 1
    assert torch.allclose(lambdas.sum(), torch.tensor(1.0, device=DEVICE), atol=1e-4), \
        f"ERROR Scale weights do not sum to 1: {lambdas.sum()}"
    print(f"  v Scale weights      : {lambdas.detach().cpu().numpy().round(3)} (sum={lambdas.sum():.4f})")

    # 9. Final quality score finite
    assert torch.isfinite(Q_hat).all(),            "ERROR Q_hat contains NaN/Inf"
    print(f"  v Q_hat range        : [{Q_hat.min():.3f}, {Q_hat.max():.3f}] (finite)")

    # 10. Output shape
    assert Q_hat.shape == (B,),                    f"ERROR Q_hat shape: {Q_hat.shape}"
    print(f"  v Output shape       : {tuple(Q_hat.shape)}")

    print("\nOK All sanity checks passed.")


run_sanity_checks(model, CFG)


## 18 . Loss Functions

### MOS Regression Loss
$$\mathcal{L}_{\text{MOS}} = \frac{1}{M}\sum_n (\hat{y}_n - y_n)^2$$

### Huber option (v3)

`CFG["MOS_LOSS"] = "huber"` (default) applies Huber loss to the MOS term: quadratic for residuals below `HUBER_DELTA`, linear beyond it. Labels with occasional large errors then pull less on the weights. `"mse"` restores the v2 loss; it is ablation B.

### Ranking Loss (margin-based)
For every pair $(a,b)$ in a batch where $y_a > y_b$:
$$\mathcal{L}_{\text{rank}} = \max(0,\; m - (\hat{y}_a - \hat{y}_b))$$
Encourages the model to preserve the *relative ordering* of images, not just absolute scores.

### Degradation Loss (optional)
$$\mathcal{L}_{\text{deg}} = \frac{1}{N}\sum_i \|\mathbf{r}_i - \mathbf{r}_i^*\|_2^2$$
Automatically disabled if `HAS_DEGRADATION_GT = False`.

### PLCC Loss (optional, `LAMBDA_PLCC`)
$$\mathcal{L}_{\text{PLCC}} = 1 - r(\hat{\mathbf{y}}, \mathbf{y})$$
Pearson correlation computed over the batch; directly optimises the evaluation metric. Disabled when `LAMBDA_PLCC = 0`.

### Rank-aware MOS weighting (optional, `RANK_WEIGHT_GAMMA`)
Each sample's squared error is weighted by $w_n \propto 1 + \gamma\,|y_n - \bar{y}|/\overline{|y - \bar{y}|}$ (normalised to mean 1), so extreme-quality images count more. Disabled when `RANK_WEIGHT_GAMMA = 0`.

### Combined Loss
$$\mathcal{L} = \lambda_{\text{MOS}}\,\mathcal{L}_{\text{MOS}} + \lambda_{\text{rank}}\,\mathcal{L}_{\text{rank}} + \lambda_{\text{deg}}\,\mathcal{L}_{\text{deg}} + \lambda_{\text{PLCC}}\,\mathcal{L}_{\text{PLCC}}$$


In [ ]:
class UIQALoss(nn.Module):
    '''
    Combined loss function for weakly supervised UIQA.

    MOS term: Huber (default) or MSE, optionally rank-weighted. Samples with MOS = NaN are
    excluded from MOS and ranking losses. Degradation loss is disabled unless has_deg_gt=True.
    '''
    def __init__(
        self,
        lambda_mos   : float = 1.0,
        lambda_rank  : float = 0.5,
        lambda_deg   : float = 0.1,
        margin       : float = 0.05,
        has_deg_gt   : bool  = False,
        lambda_plcc  : float = 0.0,
        rank_weight_gamma: float = 0.0,
        mos_loss     : str   = "huber",
        huber_delta  : float = 0.1,
    ):
        super().__init__()
        assert mos_loss in ("mse", "huber"), f"unknown mos_loss {mos_loss}"
        self.lambda_mos  = lambda_mos
        self.lambda_rank = lambda_rank
        self.lambda_deg  = lambda_deg
        self.margin      = margin
        self.has_deg_gt  = has_deg_gt
        self.lambda_plcc = lambda_plcc
        self.rank_weight_gamma = rank_weight_gamma
        self.mos_loss    = mos_loss
        self.huber_delta = huber_delta

    def forward(
        self,
        Q_hat    : torch.Tensor,        # (B,)
        mos      : torch.Tensor,        # (B,)
        r_pred   : Optional[torch.Tensor] = None,   # (B, N, K)
        r_gt     : Optional[torch.Tensor] = None,   # (B, N, K) - if GT exists
    ) -> Dict[str, torch.Tensor]:

        valid = torch.isfinite(mos)                   # mask for labelled samples

        # -- MOS regression loss (Huber or MSE, optional rank weighting) --
        if valid.sum() > 0:
            y_v = mos[valid].float()
            p_v = Q_hat[valid].float()
            if self.rank_weight_gamma > 0 and valid.sum() > 1:
                # samples far from the batch-mean MOS count more; weights normalised to mean 1
                dev = (y_v - y_v.mean()).abs()
                w = 1.0 + self.rank_weight_gamma * dev / (dev.mean() + 1e-8)
                w = w / w.mean()
            else:
                w = torch.ones_like(y_v)
            if self.mos_loss == "huber":
                per = F.huber_loss(p_v, y_v, reduction="none", delta=self.huber_delta)
            else:
                per = (p_v - y_v) ** 2
            L_mos = (w * per).mean()
        else:
            L_mos = torch.tensor(0.0, device=Q_hat.device)

        # -- Ranking loss ------------------------------------------------
        L_rank = torch.tensor(0.0, device=Q_hat.device)
        if valid.sum() >= 2:
            Q_v, y_v = Q_hat[valid], mos[valid]
            pairs = [(i, j) for i in range(len(y_v)) for j in range(len(y_v)) if y_v[i] > y_v[j]]
            if pairs:
                a_idx = torch.tensor([p[0] for p in pairs], device=Q_hat.device)
                b_idx = torch.tensor([p[1] for p in pairs], device=Q_hat.device)
                L_rank = F.relu(self.margin - (Q_v[a_idx] - Q_v[b_idx])).mean()

        # -- Degradation loss (only when pseudo-label / GT targets are used) --
        L_deg = torch.tensor(0.0, device=Q_hat.device)
        if self.has_deg_gt and r_pred is not None and r_gt is not None:
            L_deg = F.mse_loss(r_pred, r_gt)

        # -- Differentiable PLCC loss: 1 - Pearson r over the batch --------
        L_plcc = torch.tensor(0.0, device=Q_hat.device)
        if self.lambda_plcc > 0 and valid.sum() >= 3:
            Q_c = Q_hat[valid].float()
            y_c = mos[valid].float()
            Q_c = Q_c - Q_c.mean()
            y_c = y_c - y_c.mean()
            corr = (Q_c * y_c).sum() / (Q_c.norm() * y_c.norm() + 1e-8)
            L_plcc = 1.0 - corr

        total = (
            self.lambda_mos  * L_mos +
            self.lambda_rank * L_rank +
            self.lambda_deg  * L_deg +
            self.lambda_plcc * L_plcc
        )

        return {
            "total" : total,
            "L_mos" : L_mos,
            "L_rank": L_rank,
            "L_deg" : L_deg,
            "L_plcc": L_plcc,
        }


def build_criterion(cfg: dict) -> UIQALoss:
    return UIQALoss(
        lambda_mos  = cfg["LAMBDA_MOS"],
        lambda_rank = cfg["LAMBDA_RANK"],
        lambda_deg  = cfg["LAMBDA_DEG"],
        margin      = cfg["RANK_MARGIN"],
        has_deg_gt  = cfg["HAS_DEGRADATION_GT"],
        lambda_plcc = cfg.get("LAMBDA_PLCC", 0.0),
        rank_weight_gamma = cfg.get("RANK_WEIGHT_GAMMA", 0.0),
        mos_loss    = cfg.get("MOS_LOSS", "huber"),
        huber_delta = cfg.get("HUBER_DELTA", 0.1),
    )


criterion = build_criterion(CFG)
print("Loss function configured.")
print(f"  MOS loss: {CFG['MOS_LOSS']} (delta={CFG['HUBER_DELTA']})")
print(f"  Degradation loss active: {CFG['HAS_DEGRADATION_GT']}")


## 19 . Optimiser, Scheduler, Scaler

In [ ]:
import copy

class ModelEMA:
    """Exponential moving average of weights. Use `.module` for validation / checkpoints."""
    def __init__(self, net: nn.Module, decay: float):
        self.module = copy.deepcopy(net).eval()
        for p_ in self.module.parameters():
            p_.requires_grad_(False)
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, net: nn.Module) -> None:
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))   # warm-up so early steps are not stuck at init
        for e, p_ in zip(self.module.parameters(), net.parameters()):
            e.mul_(d).add_(p_.detach(), alpha=1 - d)
        for e, b in zip(self.module.buffers(), net.buffers()):
            e.copy_(b)


def make_optimizer(net: nn.Module) -> torch.optim.Optimizer:
    """AdamW with a lower LR on the pretrained CNN backbone than on the new layers."""
    bb     = list(net.cnn.features.parameters())
    bb_ids = {id(p_) for p_ in bb}
    rest   = [p_ for p_ in net.parameters() if id(p_) not in bb_ids]
    return torch.optim.AdamW(
        [{"params": bb,   "lr": CFG["LR"] * CFG.get("BACKBONE_LR_MULT", 1.0)},
         {"params": rest, "lr": CFG["LR"]}],
        weight_decay=CFG["WEIGHT_DECAY"],
    )


optimizer = make_optimizer(model)

# Cosine annealing with warm restarts for better convergence over 200 epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=50, T_mult=2, eta_min=CFG["LR"] * 1e-2
)

scaler = GradScaler(enabled=CFG["USE_AMP"] and DEVICE.type=="cuda")
ema    = ModelEMA(model, CFG["EMA_DECAY"])

print(f"Optimiser  : AdamW  head lr={CFG['LR']}  backbone lr={CFG['LR']*CFG.get('BACKBONE_LR_MULT',1.0):g}  wd={CFG['WEIGHT_DECAY']}")
print(f"Scheduler  : CosineAnnealingWarmRestarts  T_0=50  T_mult=2")
print(f"EMA decay  : {CFG['EMA_DECAY']}")
print(f"Mixed prec : {CFG['USE_AMP'] and DEVICE.type=='cuda'}")


## 20 . Training Loop

### Weakly Supervised Multiple-Instance Learning

The dataset provides only **image-level MOS**. No patch-level quality labels exist.

The model treats:
- **Image = Bag**  
- **Patches = Instances**

Image-level MOS supervises the aggregated patch predictions $\hat{y}_n = \sum_i \alpha_{n,i} q_{n,i}$.  
The patch-level quality estimates and importance weights are learned as **latent variables** through backpropagation.


In [ ]:
def mixup_batch(imgs, mos, alpha=0.3):
    """Apply Mixup augmentation to a batch. Returns mixed images and MOS."""
    if alpha <= 0:
        return imgs, mos
    lam = float(np.random.beta(alpha, alpha))
    B = imgs.size(0)
    idx = torch.randperm(B, device=imgs.device)
    mixed_imgs = lam * imgs + (1 - lam) * imgs[idx]
    mixed_mos  = lam * mos  + (1 - lam) * mos[idx]
    return mixed_imgs, mixed_mos


def train_epoch(
    model     : nn.Module,
    loader    : DataLoader,
    optimizer : torch.optim.Optimizer,
    criterion : UIQALoss,
    scaler    : GradScaler,
    cfg       : dict,
    ema       = None,
) -> Dict[str, float]:
    model.train()
    # model.train() also switches frozen BatchNorm layers to train mode, which would keep
    # updating their running stats. While the pretrained CNN is frozen, keep them in eval mode.
    _cnn = getattr(model, "cnn", None)
    if _cnn is not None and hasattr(_cnn, "features") and \
            not any(p_.requires_grad for p_ in _cnn.features.parameters()):
        for m_ in _cnn.features.modules():
            if isinstance(m_, nn.BatchNorm2d):
                m_.eval()
    totals = {"total":0., "L_mos":0., "L_rank":0., "L_deg":0., "L_plcc":0.}
    n_batches = 0

    for imgs, mos in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        mos  = mos.to(DEVICE,  non_blocking=True)

        # imgs, mos = mixup_batch(imgs, mos, alpha=0.3)  # Disabled: hurts on small dataset

        optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
              raw_imgs = imgs * torch.tensor(IMAGENET_STD, device=imgs.device).view(1,3,1,1) \
                         + torch.tensor(IMAGENET_MEAN, device=imgs.device).view(1,3,1,1)
              raw_imgs = raw_imgs.clamp(0, 1)
              r_gt = compute_degradation_pseudolabels(raw_imgs, cfg["PATCH_SIZES"])

        with autocast(enabled=cfg["USE_AMP"] and DEVICE.type=="cuda"):
              out  = model(imgs)
              loss_dict = criterion(out["Q_hat"], mos, r_pred=out["r"], r_gt=r_gt)

        scaler.scale(loss_dict["total"]).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg["GRAD_CLIP"])
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)

        for k, v in loss_dict.items():
            totals[k] += v.item()
        n_batches += 1

    return {k: v/max(n_batches,1) for k,v in totals.items()}


def validate_epoch(
    model     : nn.Module,
    loader    : DataLoader,
    criterion : UIQALoss,
    cfg       : dict,
) -> Tuple[Dict[str,float], np.ndarray, np.ndarray]:
    model.eval()
    totals = {"total":0., "L_mos":0., "L_rank":0., "L_deg":0., "L_plcc":0.}
    all_pred, all_gt = [], []
    n_batches = 0

    with torch.no_grad():
        for imgs, mos in tqdm(loader, desc="Val  ", leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            mos  = mos.to(DEVICE,  non_blocking=True)

            with torch.no_grad():
                  raw_imgs = imgs * torch.tensor(IMAGENET_STD, device=imgs.device).view(1,3,1,1) \
                             + torch.tensor(IMAGENET_MEAN, device=imgs.device).view(1,3,1,1)
                  raw_imgs = raw_imgs.clamp(0, 1)
                  r_gt = compute_degradation_pseudolabels(raw_imgs, cfg["PATCH_SIZES"])

            with autocast(enabled=cfg["USE_AMP"] and DEVICE.type=="cuda"):
                  out  = model(imgs)
                  loss_dict = criterion(out["Q_hat"], mos, r_pred=out["r"], r_gt=r_gt)

            for k, v in loss_dict.items():
                totals[k] += v.item()
            n_batches += 1

            pred = out["Q_hat"].cpu().numpy()
            gt   = mos.cpu().numpy()
            valid = np.isfinite(gt)
            if valid.any():
                all_pred.append(pred[valid])
                all_gt.append(gt[valid])

    losses = {k: v/max(n_batches,1) for k,v in totals.items()}
    preds  = np.concatenate(all_pred) if all_pred else np.array([])
    gts    = np.concatenate(all_gt)   if all_gt   else np.array([])
    return losses, preds, gts


## 21 . Evaluation Metrics

$$\text{PLCC} = \frac{\sum_i(\hat{y}_i - \bar{\hat{y}})(y_i - \bar{y})}{\sqrt{\sum_i(\hat{y}_i-\bar{\hat{y}})^2 \sum_i(y_i-\bar{y})^2}}$$

$$\text{SROCC} = 1 - \frac{6\sum_i d_i^2}{N(N^2-1)}$$

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_i(y_i - \hat{y}_i)^2}$$

All metrics are only computed when MOS labels are available.


In [ ]:
def compute_metrics(preds: np.ndarray, gts: np.ndarray) -> Dict[str, float]:
    '''
    Compute PLCC, SROCC, KRCC (Kendall Rank Correlation Coefficient), RMSE.
    Returns nan for all metrics if insufficient data.
    '''
    if len(preds) < 2:
        print("WARNING  Insufficient labelled samples for metric computation.")
        return {"PLCC": float("nan"), "SROCC": float("nan"),
                "KRCC": float("nan"), "RMSE": float("nan")}

    plcc,  _ = pearsonr(preds, gts)
    srocc, _ = spearmanr(preds, gts)
    krcc,  _ = kendalltau(preds, gts)
    rmse     = float(np.sqrt(np.mean((preds - gts)**2)))

    return {
        "PLCC": float(plcc),
        "SROCC": float(srocc),
        "KRCC": float(krcc),
        "RMSE": rmse,
    }


## 22 . Checkpointing

In [ ]:
def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    metrics: dict,
    path: str,
) -> None:
    torch.save({
        "epoch"     : epoch,
        "state_dict": model.state_dict(),
        "optimizer" : optimizer.state_dict(),
        "metrics"   : metrics,
        "cfg"       : CFG,
        "seed"      : CFG["SEED"],
    }, path)


def load_checkpoint(model: nn.Module, path: str, device: torch.device) -> dict:
    '''Load checkpoint. Returns metadata dict.'''
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}  metrics: {ckpt['metrics']}")
    return ckpt


## 23 . Main Training Loop

**Prerequisites:**  
Supervised training requires MOS labels. If no labels are available, this cell prints a clear message and skips.  
No labels are fabricated.


In [ ]:
def check_labels_available(df: pd.DataFrame) -> bool:
    has_labels = df["mos"].notna().any()
    if not has_labels:
        print("WARNING  No MOS labels found.")
        print("   Supervised training, PLCC, SROCC, and RMSE require human quality labels.")
        print("   You can still run inference and visualization.")
    return has_labels

HAS_LABELS = check_labels_available(df_all)

# Re-evaluate loaders to ensure they are built with the correct df_all
loaders = build_loaders(df_all, CFG)
for s, ld in loaders.items():
    print(f"  {s:5s}: {len(ld.dataset)} images, {len(ld)} batches")

In [ ]:
def compute_degradation_pseudolabels(raw_images: torch.Tensor, patch_sizes: List[int]) -> torch.Tensor:
    """
    raw_images: (B, 3, H, W), values in [0,1], NOT ImageNet-normalized.
    Returns: (B, N_total, 5) pseudo-labels in the SAME patch order the model uses
             (iterated by patch_size, matching extract_patches_multiscale's order).
    Attributes: [color_cast, haze, blur, contrast, noise]
    """
    B = raw_images.shape[0]
    all_labels = []

    for p in patch_sizes:
        patches = F.unfold(raw_images, kernel_size=p, stride=p)      # (B, 3*p*p, N)
        N = patches.shape[-1]
        patches = patches.permute(0, 2, 1).reshape(B, N, 3, p, p)     # (B, N, 3, p, p)

        r, g, b = patches[:,:,0], patches[:,:,1], patches[:,:,2]
        gray = (r + g + b) / 3.0

        # 1. Color cast: max channel-mean deviation from gray-mean
        mean_r, mean_g, mean_b = r.mean((-1,-2)), g.mean((-1,-2)), b.mean((-1,-2))
        mean_gray = gray.mean((-1,-2))
        color_cast = torch.stack([
            (mean_r - mean_gray).abs(),
            (mean_g - mean_gray).abs(),
            (mean_b - mean_gray).abs(),
        ], dim=-1).max(dim=-1).values                                  # (B, N)

        # 2. Haze: 1 - dark channel (min over RGB, then min over patch)
        dark_channel = torch.minimum(torch.minimum(r, g), b)
        dark_channel = dark_channel.amin(dim=(-1,-2))                  # (B, N)
        haze = 1.0 - dark_channel

        # 3. Blur: inverse of Laplacian variance
        lap_kernel = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]], dtype=raw_images.dtype, device=raw_images.device)
        lap_kernel = lap_kernel.view(1,1,3,3)
        gray_flat = gray.reshape(B*N, 1, p, p)
        lap = F.conv2d(gray_flat, lap_kernel, padding=1)
        lap_var = lap.reshape(B, N, -1).var(dim=-1)
        blur = 1.0 / (1.0 + lap_var)

        # 4. Contrast: 1 - normalized std
        contrast = 1.0 - gray.reshape(B, N, -1).std(dim=-1) / 0.5

        # 5. Noise: high-frequency energy fraction (Laplacian energy vs total)
        total_energy = (gray_flat ** 2).reshape(B, N, -1).sum(dim=-1) + 1e-6
        hf_energy = (lap ** 2).reshape(B, N, -1).sum(dim=-1)
        noise = (hf_energy / total_energy).clamp(0, 1)

        label = torch.stack([color_cast, haze, blur, contrast, noise], dim=-1)  # (B, N, 5)
        all_labels.append(label)

    return torch.cat(all_labels, dim=1)   # (B, N_total, 5) — same scale order as model

## 22b . Optional: KADID-10k IQA-Domain Pretraining

If `CFG["KADID_DIR"]` points to a KADID-10k folder (`images/` + `dmos.csv`), the model is first trained on it
(DMOS rescaled from [1,5] to [0,1]) and then fine-tuned on UWIQA with a fresh optimiser and scheduler.
Pretrained weights are cached in `checkpoints/kadid_pretrained.pt` and reused on re-runs.
If `KADID_DIR` is `None` this cell does nothing.

In [ ]:
def kadid_ckpt_path(cfg) -> Path:
    '''One KADID checkpoint per backbone and image size (weights are not interchangeable).'''
    return Path(cfg["CHECKPOINT_DIR"]) / f"kadid_{cfg['CNN_BACKBONE']}_{cfg['IMAGE_SIZE']}.pt"


def load_compatible(net: nn.Module, state: dict, label: str) -> None:
    '''Load only tensors whose name and shape match, so a checkpoint from another architecture cannot crash.'''
    own = net.state_dict()
    ok = {k: v for k, v in state.items() if k in own and own[k].shape == v.shape}
    net.load_state_dict(ok, strict=False)
    print(f"  loaded {len(ok)}/{len(own)} tensors from {label}")


def build_kadid_df(kadid_dir: str, csv_path: Optional[str] = None) -> pd.DataFrame:
    root  = Path(kadid_dir)
    csv_p = Path(csv_path) if csv_path else root / "dmos.csv"
    ann   = pd.read_csv(csv_p)
    for col in ("dist_img", "dmos"):
        if col not in ann.columns:
            raise ValueError(f"{csv_p} has no '{col}' column; found {list(ann.columns)}")
    img_dir = root / "images"
    df = pd.DataFrame({
        "image_name": ann["dist_img"],
        "image_path": [str(img_dir / n) for n in ann["dist_img"]],
        "mos"       : (ann["dmos"].astype(float) - 1.0) / 4.0,    # [1,5] -> [0,1]
    })
    return df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)


def pretrain_on_kadid(net: nn.Module, cfg: dict) -> None:
    from torch.utils.data import RandomSampler
    df = build_kadid_df(cfg["KADID_DIR"], cfg.get("KADID_CSV"))
    if len(df) == 0:
        raise RuntimeError(f"No KADID images found under {cfg['KADID_DIR']}/images")
    print(f"KADID pretraining on {len(df)} images")
    # train=False: no colour/crop augmentation, it would change the quality label
    ds = UWIQADataset(df, cfg["IMAGE_SIZE"], train=False)
    sampler = RandomSampler(ds, replacement=True, num_samples=cfg["PRETRAIN_ITERS"] * cfg["BATCH_SIZE"])
    ld = DataLoader(ds, batch_size=cfg["BATCH_SIZE"], sampler=sampler, num_workers=0,
                    pin_memory=(DEVICE.type == "cuda"), drop_last=True)
    opt   = torch.optim.AdamW(net.parameters(), lr=cfg["PRETRAIN_LR"], weight_decay=cfg["WEIGHT_DECAY"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["PRETRAIN_EPOCHS"])
    sc    = GradScaler(enabled=cfg["USE_AMP"] and DEVICE.type == "cuda")
    for ep in range(1, cfg["PRETRAIN_EPOCHS"] + 1):
        losses = train_epoch(net, ld, opt, criterion, sc, cfg)
        sched.step()
        print(f"[KADID pretrain] epoch {ep:02d}/{cfg['PRETRAIN_EPOCHS']}  loss={losses['total']:.4f}")
    kp = kadid_ckpt_path(cfg)
    torch.save({"state_dict": net.state_dict()}, kp)
    print(f"Saved KADID-pretrained weights to {kp}")


if CFG.get("KADID_DIR") and Path(CFG["KADID_DIR"]).exists():
    # Degradation estimator first (stage 1), so KADID fine-tunes from it
    if model.use_film and deg_ckpt_path(CFG).exists():
        model.deg_est.load_state_dict(torch.load(deg_ckpt_path(CFG), map_location=DEVICE)["state_dict"])
    kp = kadid_ckpt_path(CFG)
    if kp.exists():
        load_compatible(model, torch.load(kp, map_location=DEVICE)["state_dict"], kp.name)
    else:
        pretrain_on_kadid(model, CFG)
    # Fresh optimiser / scheduler for fine-tuning on UWIQA
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=50, T_mult=2, eta_min=CFG["LR"] * 1e-2)
else:
    print('KADID pretraining skipped (set CFG["KADID_DIR"] to a KADID-10k folder to enable).')


In [ ]:
shistory = {"train_loss":[], "val_loss":[], "PLCC":[], "SROCC":[], "KRCC":[], "RMSE":[]}
best_srocc    = -float("inf")
best_epoch    = 0
patience_ctr  = 0
CKPT_BEST     = Path(CFG["CHECKPOINT_DIR"]) / "best_model.pt"
CKPT_LAST     = Path(CFG["CHECKPOINT_DIR"]) / "last_model.pt"

if HAS_LABELS and "train" in loaders and "val" in loaders and CFG["KFOLD"] < 2:
    print(f"Starting training for {CFG['NUM_EPOCHS']} epochs...")
    print(f"Best-model selection metric: SROCC")
    print(f"Early stopping patience: {CFG['EARLY_STOP_PAT']} epochs\n")

    # Freeze the pretrained CNN for the first FREEZE_EPOCHS epochs to limit overfitting
    FREEZE_EP = CFG.get("FREEZE_EPOCHS", 20)
    if FREEZE_EP > 0:
        for param in model.cnn.features.parameters():
            param.requires_grad = False
        print(f"Pretrained CNN backbone frozen for first {FREEZE_EP} epochs.")

    _csv = Path(CFG["OUTPUT_DIR"]) / "logs" / "single_split_epochs.csv"
    _csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(columns=["epoch","train_loss","val_loss","PLCC","SROCC","KRCC","RMSE","lr","best_SROCC","is_best"]).to_csv(_csv, index=False)

    for epoch in range(1, CFG["NUM_EPOCHS"]+1):
        # Unfreeze backbone after the freeze period
        if FREEZE_EP > 0 and epoch == FREEZE_EP + 1:
            for param in model.cnn.features.parameters():
                param.requires_grad = True
            print(f"\n  ** Backbone unfrozen at epoch {epoch} **")

        # -- Train -----------------------------------------------------
        train_losses = train_epoch(model, loaders["train"], optimizer, criterion, scaler, CFG, ema=ema)
        _lr = optimizer.param_groups[-1]["lr"]
        scheduler.step()

        # -- Validate --------------------------------------------------
        val_losses, val_preds, val_gts = validate_epoch(ema.module, loaders["val"], criterion, CFG)   # EMA weights
        metrics = compute_metrics(val_preds, val_gts)

        # -- Logging ---------------------------------------------------
        shistory["train_loss"].append(train_losses["total"])
        shistory["val_loss"].append(val_losses["total"])
        shistory["PLCC"].append(metrics["PLCC"])
        shistory["SROCC"].append(metrics["SROCC"])
        shistory["KRCC"].append(metrics["KRCC"])
        shistory["RMSE"].append(metrics["RMSE"])

        print(
            f"Epoch {epoch:03d}/{CFG['NUM_EPOCHS']}  "
            f"train_loss={train_losses['total']:.4f}  "
            f"val_loss={val_losses['total']:.4f}  "
            f"PLCC={metrics['PLCC']:.4f}  "
            f"SROCC={metrics['SROCC']:.4f}  "
            f"KRCC={metrics['KRCC']:.4f}  "
            f"RMSE={metrics['RMSE']:.4f}"
        )

        # -- Best model checkpoint -------------------------------------
        if np.isfinite(metrics["SROCC"]) and metrics["SROCC"] > best_srocc:
            best_srocc = metrics["SROCC"]
            best_epoch = epoch
            patience_ctr = 0
            save_checkpoint(ema.module, optimizer, epoch, metrics, str(CKPT_BEST))
            print(f"  * New best SROCC={best_srocc:.4f}  saved to {CKPT_BEST}")
        else:
            patience_ctr += 1

        save_checkpoint(ema.module, optimizer, epoch, metrics, str(CKPT_LAST))

        pd.DataFrame([{"epoch": epoch, "train_loss": train_losses["total"], "val_loss": val_losses["total"],
                       "PLCC": metrics["PLCC"], "SROCC": metrics["SROCC"], "KRCC": metrics["KRCC"],
                       "RMSE": metrics["RMSE"], "lr": _lr, "best_SROCC": best_srocc,
                       "is_best": best_epoch == epoch}]).to_csv(_csv, mode="a", header=False, index=False)
        if CFG.get("EARLY_STOP_PAT", 0) > 0 and epoch - max(best_epoch, FREEZE_EP) >= CFG["EARLY_STOP_PAT"]:
            print(f"Early stopping at epoch {epoch} (patience {CFG['EARLY_STOP_PAT']}).")
            break

    print(f"\nTraining complete. Best SROCC = {best_srocc:.4f} at epoch {best_epoch}")

else:
    print("Single-split training skipped (KFOLD >= 2 uses the k-fold runner, or no labels).")
    print("Load a dataset with MOS annotations to enable training.")

In [ ]:
def plot_training_history(history: dict, save_dir: str) -> None:
    if not any(history["train_loss"]):
        print("No training history to plot.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(history["PLCC"],  label="PLCC")
    axes[1].plot(history["SROCC"], label="SROCC")
    if "KRCC" in history and history["KRCC"]:
        axes[1].plot(history["KRCC"], label="KRCC")
    axes[1].set_title("Correlation"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    axes[2].plot(history["RMSE"])
    axes[2].set_title("RMSE"); axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.savefig(Path(save_dir) / "training_curves.png", dpi=120)
    plt.show()

plot_training_history(shistory, CFG["OUTPUT_DIR"])

## 24 . Test Evaluation

In [ ]:
def evaluate_on_test(
    model     : nn.Module,
    loader    : DataLoader,
    criterion : UIQALoss,
    cfg       : dict,
    ckpt_path : str,
) -> Dict[str, float]:
    if not Path(ckpt_path).exists():
        print(f"No checkpoint at {ckpt_path}. Evaluating current model weights.")
    else:
        load_checkpoint(model, ckpt_path, DEVICE)

    losses, preds, gts = validate_epoch(model, loader, criterion, cfg)
    metrics = compute_metrics(preds, gts)

    print("\n-- Test Evaluation --------------------------------------")
    if np.isfinite(metrics["PLCC"]):
        for k, v in metrics.items():
            print(f"  {k:12s}: {v:.4f}")
    else:
        print("  No MOS labels available - metrics cannot be computed.")
        print("  Predictions were generated successfully.")
    print("---------------------------------------------------------")
    return metrics, preds, gts


if HAS_LABELS and "test" in loaders and CFG["KFOLD"] < 2:
    test_metrics, test_preds, test_gts = evaluate_on_test(
        model, loaders["test"], criterion, CFG,
        str(CKPT_BEST) if Path(CKPT_BEST).exists() else ""
    )
else:
    print("Test evaluation skipped: no labels or no test split.")
    test_metrics, test_preds, test_gts = {}, np.array([]), np.array([])


### Test-Time Augmentation (TTA)

Averages `Q_hat` over 6 deterministic views (identity, H-flip, V-flip, H+V flip, 90% centre crop, and its H-flip).
Controlled by `CFG["USE_TTA"]`. The logistic fit below also uses TTA predictions when enabled, so the
fitted mapping and the test predictions are computed the same way.

In [ ]:
@torch.no_grad()
def tta_batch_predict(net: nn.Module, imgs: torch.Tensor, cfg: dict, crop_frac: float = 0.9) -> torch.Tensor:
    '''Average Q_hat over 4 flip views (+2 crop views if CFG["TTA_CROP"]). imgs: (B,3,H,W) already on DEVICE.'''
    net.eval()
    H = imgs.shape[-1]
    c = int(round(H * crop_frac)); o = (H - c) // 2
    crop = F.interpolate(imgs[:, :, o:o+c, o:o+c], size=(H, H), mode="bilinear", align_corners=False)
    views = [imgs, torch.flip(imgs, dims=[3]), torch.flip(imgs, dims=[2]),
             torch.flip(imgs, dims=[2, 3])]
    if cfg.get("TTA_CROP", False):   # crop views resample the image and lowered SROCC; off by default
        views += [crop, torch.flip(crop, dims=[3])]
    preds = []
    for v in views:
        with autocast(enabled=cfg["USE_AMP"] and DEVICE.type == "cuda"):
            preds.append(net(v)["Q_hat"].float())
    return torch.stack(preds, dim=0).mean(dim=0)


@torch.no_grad()
def predict_scores(loader: DataLoader, use_tta=None, net=None):
    '''Returns (preds, gts) for labelled samples. Uses TTA when CFG["USE_TTA"] (or use_tta=True).'''
    use_tta = CFG.get("USE_TTA", False) if use_tta is None else use_tta
    net = model if net is None else net
    net.eval()
    preds, gts = [], []
    for imgs, mos in tqdm(loader, desc="TTA" if use_tta else "Eval", leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        if use_tta:
            q = tta_batch_predict(net, imgs, CFG)
        else:
            with autocast(enabled=CFG["USE_AMP"] and DEVICE.type == "cuda"):
                q = net(imgs)["Q_hat"].float()
        q = q.cpu().numpy(); y = mos.numpy()
        ok = np.isfinite(y)
        preds.append(q[ok]); gts.append(y[ok])
    return (np.concatenate(preds) if preds else np.array([]),
            np.concatenate(gts)   if gts   else np.array([]))


# Non-augmented, full-set loader for the TRAIN split (used by the logistic fit)
if HAS_LABELS and "train" in loaders:
    train_eval_loader = DataLoader(
        UWIQADataset(df_all[df_all["split"] == "train"], CFG["IMAGE_SIZE"], train=False),
        batch_size=CFG["BATCH_SIZE"], shuffle=False, num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

# Plain vs TTA on the test split (the best checkpoint is already loaded by the cell above)
if HAS_LABELS and "test" in loaders and CFG.get("USE_TTA", False) and CFG["KFOLD"] < 2:
    test_preds_plain, test_gts_plain = test_preds.copy(), test_gts.copy()
    test_preds, test_gts = predict_scores(loaders["test"], use_tta=True)
    m_plain = compute_metrics(test_preds_plain, test_gts_plain)
    m_tta   = compute_metrics(test_preds, test_gts)
    print("\n-- TEST metrics (raw, before logistic fit): plain vs TTA ------")
    print(f"  {'metric':8s}{'plain':>10s}{'TTA':>10s}")
    for key in ["PLCC", "SROCC", "KRCC", "RMSE"]:
        print(f"  {key:8s}{m_plain[key]:10.4f}{m_tta[key]:10.4f}")
    test_metrics = m_tta


## 21b . Nonlinear (5-Parameter Logistic) Regression Fit (v3)

A monotonic 5-parameter logistic (VQEG-style) maps raw scores to the MOS scale:

$$Q_{fit}(x) = \beta_1\left(\frac{1}{2} - \frac{1}{1+\exp(\beta_2(x-\beta_3))}\right) + \beta_4 x + \beta_5$$

**Procedure (v3):**
- The map is fitted on **out-of-fold validation predictions** (every pool image is predicted by a model that did not train on it). It is never fitted on training predictions, which are in-sample.
- 12 random restarts of `curve_fit` are run. The restart with the **lowest squared error on the fit data** is kept. Correlation is not used for selection, because that would select on the same data.
- In k-fold mode, the fitted map is applied to each fold model's test output, and the calibrated outputs are averaged.
- The test split is touched once, for the final report.


In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import f as f_dist


def logistic5(x: np.ndarray, b1: float, b2: float, b3: float, b4: float, b5: float) -> np.ndarray:
    '''5-parameter logistic (VQEG-style) nonlinear mapping from raw score x to the MOS scale.'''
    return b1 * (0.5 - 1.0 / (1.0 + np.exp(b2 * (x - b3)))) + b4 * x + b5


def _all_metrics(pred: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    '''PLCC (=LCC), SROCC, KROCC (Kendall tau), RMSE for one prediction/ground-truth pair.'''
    if len(pred) < 2:
        return {"LCC": float("nan"), "PLCC": float("nan"), "SROCC": float("nan"),
                "KROCC": float("nan"), "RMSE": float("nan")}
    lcc,   _ = pearsonr(pred, gt)
    srocc, _ = spearmanr(pred, gt)
    krocc, _ = kendalltau(pred, gt)
    rmse = float(np.sqrt(np.mean((pred - gt) ** 2)))
    return {"LCC": float(lcc), "PLCC": float(lcc), "SROCC": float(srocc),
            "KROCC": float(krocc), "RMSE": rmse}


def f_test_variances(resid_a: np.ndarray, resid_b: np.ndarray, label_a="raw", label_b="fitted") -> Dict[str, float]:
    '''Two-tailed F-test on residual variances (H0: equal). Reported for context only.'''
    var_a, var_b = np.var(resid_a, ddof=1), np.var(resid_b, ddof=1)
    df_a, df_b = len(resid_a) - 1, len(resid_b) - 1
    if var_b == 0 or df_a < 1 or df_b < 1:
        return {"F_stat": float("nan"), "p_value": float("nan"), "df1": df_a, "df2": df_b}
    F_ = var_a / var_b
    p_one = f_dist.sf(F_, df_a, df_b) if F_ >= 1 else f_dist.cdf(F_, df_a, df_b)
    return {"F_stat": float(F_), "p_value": float(2 * min(p_one, 1 - p_one)), "df1": df_a, "df2": df_b}


def fit_logistic5(x_fit: np.ndarray, y_fit: np.ndarray, n_iters: int = 12, seed: int = 42):
    '''
    Multi-restart nonlinear least-squares fit of logistic5(x_fit) -> y_fit.
    Restarts are ranked by squared error on the SAME fit data, which is the quantity the
    fit minimises. (v2 ranked restarts by correlation on a separate split, which is selection
    on the evaluation data.)
    Returns (params or None if every restart failed, log DataFrame).
    '''
    assert 10 <= n_iters <= 15, "Use 10-15 restarts."
    rng = np.random.RandomState(seed)
    x_mean, x_std = float(np.mean(x_fit)), float(np.std(x_fit) + 1e-8)
    y_mean, y_std = float(np.mean(y_fit)), float(np.std(y_fit) + 1e-8)
    best_params, best_mse, log = None, np.inf, []
    for it in range(1, n_iters + 1):
        p0 = [
            y_std * (1 + 0.5 * rng.randn()),
            (1.0 / x_std) * (1 + 0.5 * rng.randn()),
            x_mean + 0.3 * x_std * rng.randn(),
            0.1 * rng.randn(),
            y_mean + 0.3 * y_std * rng.randn(),
        ]
        try:
            params, _ = curve_fit(logistic5, x_fit, y_fit, p0=p0, maxfev=20000)
            pred = logistic5(x_fit, *params)
            if not np.all(np.isfinite(pred)):
                raise RuntimeError("non-finite fit output")
            mse = float(np.mean((pred - y_fit) ** 2))
            log.append({"iter": it, "fit_mse": mse})
            if mse < best_mse:
                best_mse, best_params = mse, np.asarray(params)
        except Exception as e:
            log.append({"iter": it, "fit_mse": np.nan, "error": str(e)[:80]})
    return best_params, pd.DataFrame(log)


# -- Single-split path (only when KFOLD < 2). The k-fold runner calls fit_logistic5 itself.
test_preds_fitted = np.array([])
if HAS_LABELS and CFG["KFOLD"] < 2 and "val" in loaders and "test" in loaders:
    # Fit on VALIDATION predictions (never train), apply once to TEST.
    val_preds_raw, val_gts = predict_scores(loaders["val"])
    beta, fit_log = fit_logistic5(val_preds_raw, val_gts, n_iters=12, seed=CFG["SEED"])
    print(fit_log.to_string(index=False))
    if beta is not None:
        print(f"Fitted b1..b5 = {np.round(beta, 4).tolist()}")
        test_preds_fitted = logistic5(test_preds, *beta)
        print("\n-- TEST raw  :", {k: round(v, 4) for k, v in _all_metrics(test_preds, test_gts).items()})
        print("-- TEST fitted:", {k: round(v, 4) for k, v in _all_metrics(test_preds_fitted, test_gts).items()})
        ftest = f_test_variances(test_preds - test_gts, test_preds_fitted - test_gts)
        print(f"F-test (residual variance raw vs fitted): F={ftest['F_stat']:.3f}, p={ftest['p_value']:.4f}")
    else:
        print("All restarts failed; no logistic map applied.")
else:
    print("Single-split logistic fit skipped (KFOLD >= 2 fits on out-of-fold predictions in the runner).")


In [ ]:
import seaborn as sns   # v2 dependency, kept as optional


def analyse_mos_intervals(gts, raw, fitted):
    '''Error breakdown by ground-truth MOS interval. Bins assume a 0-1 MOS scale.'''
    if len(gts) == 0:
        print("No test predictions to analyse.")
        return
    df_a = pd.DataFrame({"Ground_Truth_MOS": gts, "Raw_Predictions": raw, "Fitted_Predictions": fitted})
    bins = [0.2, 0.4, 0.6, 0.8, 1.0]
    labels = ["0.2-0.4 (Low)", "0.4-0.6 (Mid-Low)", "0.6-0.8 (Mid-High)", "0.8-1.0 (High)"]
    df_a["MOS_Interval"] = pd.cut(df_a["Ground_Truth_MOS"], bins=bins, labels=labels, include_lowest=True)

    error_stats = df_a.groupby("MOS_Interval", observed=False).apply(lambda g: pd.Series({
        "Count": len(g),
        "Percentage": len(g) / len(df_a) * 100,
        "Raw_MAE": np.mean(np.abs(g["Raw_Predictions"] - g["Ground_Truth_MOS"])),
        "Fitted_MAE": np.mean(np.abs(g["Fitted_Predictions"] - g["Ground_Truth_MOS"])),
        "Raw_RMSE": np.sqrt(np.mean((g["Raw_Predictions"] - g["Ground_Truth_MOS"]) ** 2)),
        "Fitted_RMSE": np.sqrt(np.mean((g["Fitted_Predictions"] - g["Ground_Truth_MOS"]) ** 2)),
    })).reset_index()

    print("=== MOS Distribution and Prediction Errors ===")
    display(error_stats)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.histplot(df_a["Ground_Truth_MOS"], bins=10, kde=True, ax=axes[0], color="skyblue")
    axes[0].set_title("Ground-Truth MOS Distribution"); axes[0].set_xlabel("Ground-Truth MOS"); axes[0].set_ylabel("Count")

    x = np.arange(len(labels)); width = 0.35
    axes[1].bar(x - width / 2, error_stats["Fitted_MAE"], width, label="Fitted MAE", color="teal")
    axes[1].bar(x + width / 2, error_stats["Fitted_RMSE"], width, label="Fitted RMSE", color="salmon")
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=15)
    axes[1].set_title("Error by MOS interval (fitted)"); axes[1].set_xlabel("MOS interval"); axes[1].legend()
    plt.tight_layout(); plt.show()


if len(test_gts) > 0:
    analyse_mos_intervals(test_gts, test_preds, test_preds_fitted)
else:
    print("Interval analysis deferred to the k-fold summary cell.")


## 21c . K-Fold Ensemble (v3: on by default)

`CFG["KFOLD"] = 5` trains five models per seed, each on four folds of the train+val pool and validated on the fifth. Test predictions are averaged over the five models. The test split is never used for model selection or calibration. The full procedure (seeds x folds) is in section 21d.


In [ ]:
from sklearn.model_selection import KFold
from contextlib import contextmanager


@contextmanager
def cfg_overrides(overrides: dict):
    '''Temporarily change CFG (and the loss built from it) for an ablation, then restore.'''
    global criterion
    saved = {k: CFG.get(k) for k in overrides}
    saved_crit = criterion
    CFG.update(overrides)
    criterion = build_criterion(CFG)
    try:
        yield CFG
    finally:
        CFG.update(saved)
        criterion = saved_crit


def ensure_deg_pretrained(pool_df: pd.DataFrame, path: Path = None) -> None:
    '''Run stage 1 on pool_df (cached per image set). Skipped when FiLM or deg-pretraining is off.'''
    if not (CFG.get("USE_FILM") and CFG.get("USE_DEG_PRETRAIN")):
        return
    est = DegradationEstimator(CFG["DEG_EMBED_DIM"]).to(DEVICE)
    pretrain_degradation_estimator(est, pool_df, CFG, path or deg_ckpt_path(CFG))


def init_model(deg_path: Path = None) -> "HybridUIQAModel":
    '''Fresh model; KADID weights (if cached for this backbone/size) then the degradation estimator.'''
    net = HybridUIQAModel(CFG).to(DEVICE)
    kp = kadid_ckpt_path(CFG)
    if CFG.get("USE_KADID_INIT", True) and kp.exists():
        load_compatible(net, torch.load(kp, map_location=DEVICE)["state_dict"], kp.name)
    dp = deg_path or deg_ckpt_path(CFG)
    if net.use_film and dp.exists():
        net.deg_est.load_state_dict(torch.load(dp, map_location=DEVICE)["state_dict"])
    return net


FIT_INFO = {}   # tag -> {"stopped_epoch", "best_epoch"}, filled by fit_model


def fit_model(net, fold_loaders, tag, epochs, freeze_epochs, crit):
    '''
    Train net on fold_loaders['train']; keep the best epoch by val SROCC (EMA weights).
    Every epoch: metrics are computed and printed, one row is appended to
    outputs/logs/<tag>_epochs.csv, and the EMA weights are saved to checkpoints/<tag>_last.pt.
    The best epoch is saved to checkpoints/<tag>_best.pt. Early stopping uses CFG["EARLY_STOP_PAT"].
    Returns (best_srocc, best_ckpt_path). Test data is never touched here.
    '''
    opt   = make_optimizer(net)
    ema_  = ModelEMA(net, CFG["EMA_DECAY"])
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=50, T_mult=2, eta_min=CFG["LR"] * 1e-2)
    sc    = GradScaler(enabled=CFG["USE_AMP"] and DEVICE.type == "cuda")
    ckpt_dir  = Path(CFG["CHECKPOINT_DIR"])
    ckpt      = ckpt_dir / f"{tag}_best.pt"
    ckpt_last = ckpt_dir / f"{tag}_last.pt"
    log_dir   = Path(CFG["OUTPUT_DIR"]) / "logs"; log_dir.mkdir(parents=True, exist_ok=True)
    csv_path  = log_dir / f"{tag}_epochs.csv"
    cols = ["epoch", "train_loss", "val_loss", "PLCC", "SROCC", "KRCC", "RMSE",
            "lr_backbone", "lr_head", "best_SROCC", "best_epoch", "is_best", "epochs_since_best"]
    pd.DataFrame(columns=cols).to_csv(csv_path, index=False)
    patience = CFG.get("EARLY_STOP_PAT", 0)
    best, best_ep = -float("inf"), 0

    if freeze_epochs > 0:
        for p_ in net.cnn.features.parameters():
            p_.requires_grad = False
    for ep in range(1, epochs + 1):
        if freeze_epochs > 0 and ep == freeze_epochs + 1:
            for p_ in net.cnn.features.parameters():
                p_.requires_grad = True
        tr = train_epoch(net, fold_loaders["train"], opt, crit, sc, CFG, ema=ema_)
        lr_bb, lr_hd = opt.param_groups[0]["lr"], opt.param_groups[1]["lr"]   # LRs used this epoch
        sched.step()
        vl, vp, vg = validate_epoch(ema_.module, fold_loaders["val"], crit, CFG)
        m = compute_metrics(vp, vg)

        is_best = bool(np.isfinite(m["SROCC"]) and m["SROCC"] > best)
        if is_best:
            best, best_ep = m["SROCC"], ep
            torch.save({"state_dict": ema_.module.state_dict(), "epoch": ep, "metrics": m}, ckpt)
        torch.save({"state_dict": ema_.module.state_dict(), "epoch": ep, "metrics": m}, ckpt_last)
        if CFG.get("SAVE_EPOCH_CKPTS", False):
            torch.save({"state_dict": ema_.module.state_dict(), "epoch": ep, "metrics": m},
                       ckpt_dir / f"{tag}_ep{ep:03d}.pt")

        # Patience counts epochs since the best epoch, but not before the backbone has been unfrozen
        since = ep - max(best_ep, freeze_epochs)
        pd.DataFrame([{
            "epoch": ep, "train_loss": tr["total"], "val_loss": vl["total"],
            "PLCC": m["PLCC"], "SROCC": m["SROCC"], "KRCC": m["KRCC"], "RMSE": m["RMSE"],
            "lr_backbone": lr_bb, "lr_head": lr_hd, "best_SROCC": best, "best_epoch": best_ep,
            "is_best": is_best, "epochs_since_best": since,
        }]).to_csv(csv_path, mode="a", header=False, index=False)

        print(f"  [{tag}] epoch {ep:03d}/{epochs}  train_loss={tr['total']:.4f}  val_loss={vl['total']:.4f}  "
              f"PLCC={m['PLCC']:.4f}  SROCC={m['SROCC']:.4f}  KRCC={m['KRCC']:.4f}  RMSE={m['RMSE']:.4f}  "
              f"best SROCC={best:.4f} (ep {best_ep})  no-gain {max(since, 0)}/{patience}"
              + ("  *best*" if is_best else ""))

        if patience > 0 and since >= patience:
            print(f"  [{tag}] early stopping at epoch {ep}: no val-SROCC gain for {patience} epochs "
                  f"(best {best:.4f} at epoch {best_ep}).")
            break
    FIT_INFO[tag] = {"stopped_epoch": ep, "best_epoch": best_ep}
    if not ckpt.exists():
        torch.save({"state_dict": ema_.module.state_dict()}, ckpt)
    print(f"  [{tag}] per-epoch log: {csv_path}")
    return best, ckpt


ITER_ROWS = []   # one row per finished iteration -> outputs/logs/iterations_summary.csv


def run_experiment(seed: int) -> Dict:
    '''
    One full protocol run for one training seed.
      * Data split (stratified train/val/test) and stratified k-fold partition are FIXED (CFG["SEED"]).
      The degradation estimator is pretrained inside each fold on that fold's training images only.
      * KFOLD >= 2: k models, each trained on k-1 folds of train+val and validated on the
        held-out fold. Out-of-fold (OOF) predictions over the whole pool fit the calibration.
        Test predictions = mean over the k fold models (each calibrated, then averaged).
      * KFOLD < 2: one train/val run.
    '''
    set_seed(seed)
    k = CFG["KFOLD"]
    pool = df_all[df_all["split"].isin(["train", "val"]) & df_all["mos"].notna()].reset_index(drop=True)
    test_df = df_all[(df_all["split"] == "test") & df_all["mos"].notna()].reset_index(drop=True)

    if k >= 2:
        folds = []
        for tr_idx, va_idx in make_folds(pool, k, CFG["SEED"]):
            tr = pool.iloc[tr_idx].copy(); tr["split"] = "train"
            va = pool.iloc[va_idx].copy(); va["split"] = "val"
            folds.append((tr, va))
    else:
        tr = df_all[(df_all["split"] == "train") & df_all["mos"].notna()].copy()
        va = df_all[(df_all["split"] == "val") & df_all["mos"].notna()].copy()
        folds = [(tr, va)]

    te = test_df.copy(); te["split"] = "test"
    val_preds, val_gts, val_names = [], [], []
    test_raw, fold_test_metrics = [], []
    test_gt = None

    for f, (tr, va) in enumerate(folds, 1):
        print(f"\n== seed {seed} | fold {f}/{len(folds)} | train={len(tr)} val={len(va)} test={len(te)} ==")
        dp = deg_ckpt_path(CFG, f"s{seed}_f{f}")
        ensure_deg_pretrained(tr, dp)          # this fold's training images only: no validation or test exposure
        fl = build_loaders(pd.concat([tr, va, te], ignore_index=True), CFG)
        net = init_model(dp)
        _, ck = fit_model(net, fl, f"s{seed}_f{f}", CFG["NUM_EPOCHS"], CFG["FREEZE_EPOCHS"], criterion)
        net.load_state_dict(torch.load(ck, map_location=DEVICE)["state_dict"])

        vp, vg = predict_scores(fl["val"], net=net)
        tp, tg = predict_scores(fl["test"], net=net)
        val_preds.append(vp); val_gts.append(vg); val_names.append(va["image_name"].values)
        test_raw.append(tp); test_gt = tg
        fold_test_metrics.append(compute_metrics(tp, tg))

        # ---- end of this iteration: patience ran out (or NUM_EPOCHS reached); now calculate ----
        vm, tm = compute_metrics(vp, vg), fold_test_metrics[-1]
        info = FIT_INFO.get(f"s{seed}_f{f}", {})
        print(f"\n-- ITERATION {f}/{len(folds)} (seed {seed}) finished: stopped at epoch "
              f"{info.get('stopped_epoch')}, best epoch {info.get('best_epoch')}")
        print(f"   VAL  (best checkpoint): PLCC={vm['PLCC']:.4f}  SROCC={vm['SROCC']:.4f}  KRCC={vm['KRCC']:.4f}  RMSE={vm['RMSE']:.4f}")
        print(f"   TEST (this model alone, raw): PLCC={tm['PLCC']:.4f}  SROCC={tm['SROCC']:.4f}  KRCC={tm['KRCC']:.4f}  RMSE={tm['RMSE']:.4f}")
        ITER_ROWS.append({"iteration": f, "seed": seed, "n_train": len(tr), "n_val": len(va), "n_test": len(te),
                          "stopped_epoch": info.get("stopped_epoch"), "best_epoch": info.get("best_epoch"),
                          **{f"val_{k}": vm[k] for k in ("PLCC", "SROCC", "KRCC", "RMSE")},
                          **{f"test_{k}": tm[k] for k in ("PLCC", "SROCC", "KRCC", "RMSE")}})
        pd.DataFrame(ITER_ROWS).to_csv(Path(CFG["OUTPUT_DIR"]) / "logs" / "iterations_summary.csv", index=False)
        del net
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    # -- Calibration: fit on out-of-fold (or val) predictions only --------
    x_fit = np.concatenate(val_preds)
    y_fit = np.concatenate(val_gts)
    n_fit = np.concatenate(val_names)
    beta, cal_log = fit_logistic5(x_fit, y_fit, n_iters=12, seed=seed)

    tp_mean = np.mean(test_raw, axis=0)
    if beta is not None:
        tp_cal = np.mean([logistic5(tp, *beta) for tp in test_raw], axis=0)
        oof_cal = logistic5(x_fit, *beta)
    else:
        tp_cal, oof_cal = tp_mean, x_fit

    te_names = te["image_name"].values
    return {
        "seed": seed,
        "beta": beta,
        "test_raw": _all_metrics(tp_mean, test_gt),
        "test_cal": _all_metrics(tp_cal, test_gt),
        "oof_raw":  _all_metrics(x_fit, y_fit),
        "oof_cal":  _all_metrics(oof_cal, y_fit),
        "fold_test_metrics": fold_test_metrics,
        "test_names": te_names, "test_gt": test_gt,
        "test_raw_pred": tp_mean, "test_cal_pred": tp_cal,
        "oof_names": n_fit, "oof_gt": y_fit,
        "oof_raw_pred": x_fit, "oof_cal_pred": oof_cal,
        "cal_fit_log": cal_log,
    }


def run_all_seeds(seeds) -> List[Dict]:
    out = []
    for s in seeds:
        print(f"\n################  seed {s}  ################")
        out.append(run_experiment(s))
    return out


def summarise_runs(results: List[Dict], key: str) -> pd.DataFrame:
    '''Mean and sample std (ddof=1) across seeds for the four reported metrics.'''
    rows = []
    for m in ["PLCC", "SROCC", "KROCC", "RMSE"]:
        vals = np.array([r[key][m] for r in results], dtype=float)
        mu = float(np.mean(vals))
        sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else float("nan")
        rows.append({"metric": m, "mean": mu, "std": sd,
                     "mean±std": f"{mu:.4f} ± {sd:.4f}" if len(vals) > 1 else f"{mu:.4f}",
                     "per seed": ", ".join(f"{v:.4f}" for v in vals)})
    return pd.DataFrame(rows)


## 21d · Main experiment: seeds x k-fold, mean ± std (v3)

Each seed trains `KFOLD` models; the table reports mean and sample std (ddof=1) across seeds. The **calibrated** row is the one to compare, with the calibration fitted on out-of-fold predictions only. Expect this cell to take hours on a GPU.


In [ ]:
RESULTS = []
if HAS_LABELS and CFG.get("RUN_MAIN_EXPERIMENT", False):
    RESULTS = run_all_seeds(CFG["SEEDS"])

    raw_tbl = summarise_runs(RESULTS, "test_raw")
    cal_tbl = summarise_runs(RESULTS, "test_cal")
    oof_tbl = summarise_runs(RESULTS, "oof_cal")

    cols = ["metric", "mean±std", "per seed"]
    print(f"\n=== TEST, raw model output  (k={CFG['KFOLD']}-fold ensemble, seeds {CFG['SEEDS']}) ===")
    print(raw_tbl[cols].to_string(index=False))
    print("\n=== TEST, after OOF-fitted logistic map ===")
    print(cal_tbl[cols].to_string(index=False))
    print("\n=== OOF validation predictions, after logistic map (in-sample for the map; reference only) ===")
    print(oof_tbl[cols].to_string(index=False))

    raw_tbl.to_csv(Path(CFG["OUTPUT_DIR"]) / "test_summary_raw.csv", index=False)
    cal_tbl.to_csv(Path(CFG["OUTPUT_DIR"]) / "test_summary_calibrated.csv", index=False)
    pd.concat([pd.DataFrame({"seed": r["seed"], "image_name": r["test_names"], "mos": r["test_gt"],
                             "pred_raw": r["test_raw_pred"], "pred_cal": r["test_cal_pred"]})
               for r in RESULTS]).to_csv(Path(CFG["OUTPUT_DIR"]) / "test_predictions_all_seeds.csv", index=False)

    # Globals used by the interval analysis (seed 0 ensemble)
    r0 = RESULTS[0]
    test_gts, test_preds, test_preds_fitted = r0["test_gt"], r0["test_raw_pred"], r0["test_cal_pred"]
    analyse_mos_intervals(test_gts, test_preds, test_preds_fitted)
    print("\nSaved summary tables and per-image predictions to", CFG["OUTPUT_DIR"])
else:
    print("Main experiment skipped (needs labels and RUN_MAIN_EXPERIMENT=True).")


## 21d-2 · Check 3: why is test lower than validation?

Reads the run's own outputs (per-epoch CSVs, `ITER_ROWS`, `RESULTS`). Three questions:

1. **Selection optimism:** is the best-epoch val SROCC a peak, compared with the epochs around it?
2. **Sampling noise:** is the test SROCC within what random test-sized subsets of the out-of-fold predictions give?
3. **Split shift:** does the test MOS distribution differ from the train+val pool?

Single models are compared with single models; the test ensemble is shown separately.


In [ ]:
from scipy.stats import ks_2samp

def check3_val_vs_test(n_sub=2000, window=10, seed=0):
    rng = np.random.default_rng(seed)
    log_dir = Path(CFG["OUTPUT_DIR"]) / "logs"

    # ---- (1) selection optimism: best epoch vs the epochs around it, per iteration -------------
    rows = []
    for r in ITER_ROWS:
        tag = f"s{r['seed']}_f{r['iteration']}"
        lg = pd.read_csv(log_dir / f"{tag}_epochs.csv")
        be = int(r["best_epoch"])
        w = lg[(lg["epoch"] >= be - window) & (lg["epoch"] <= be + window)]
        rows.append({"iteration": r["iteration"], "best_epoch": be,
                     "val_SROCC_best_epoch": float(lg.loc[lg["epoch"] == be, "SROCC"].iloc[0]),
                     f"val_SROCC_mean(best+-{window})": float(w["SROCC"].mean()),
                     "test_SROCC_single_model": r["test_SROCC"]})
    t1 = pd.DataFrame(rows)
    print("== (1) Selection optimism, per iteration ==")
    print(t1.round(4).to_string(index=False))
    m = t1.drop(columns=["iteration", "best_epoch"]).mean()
    print("\nMean over iterations:"); print(m.round(4).to_string())
    optimism = m["val_SROCC_best_epoch"] - m[f"val_SROCC_mean(best+-{window})"]
    shortfall = m[f"val_SROCC_mean(best+-{window})"] - m["test_SROCC_single_model"]
    print(f"\nPart of the gap from picking the peak epoch : {optimism:+.4f}")
    print(f"Gap that remains (val around best vs test)  : {shortfall:+.4f}")

    # ---- (2) is test SROCC inside the sampling noise of out-of-fold subsets? -------------------
    if RESULTS:
        r0 = RESULTS[0]
        oof_p, oof_y = np.asarray(r0["oof_raw_pred"]), np.asarray(r0["oof_gt"])
        te_p, te_y = np.asarray(r0["test_raw_pred"]), np.asarray(r0["test_gt"])
        n_te = len(te_y)
        s_te = spearmanr(te_p, te_y)[0]
        sub = np.array([spearmanr(oof_p[ix], oof_y[ix])[0]
                        for ix in (rng.choice(len(oof_y), n_te, replace=False) for _ in range(n_sub))])
        boot = np.array([spearmanr(te_p[ix], te_y[ix])[0]
                         for ix in (rng.integers(0, n_te, n_te) for _ in range(n_sub))])
        pct = float((sub <= s_te).mean())
        print(f"\n== (2) Sampling noise ==")
        print(f"Test SROCC (ensemble of {len(ITER_ROWS)} models)      : {s_te:.4f}   95% bootstrap CI [{np.percentile(boot,2.5):.3f}, {np.percentile(boot,97.5):.3f}]")
        print(f"OOF SROCC on random {n_te}-image subsets (single models): mean {sub.mean():.4f}, 5th-95th pct [{np.percentile(sub,5):.3f}, {np.percentile(sub,95):.3f}]")
        print(f"Share of OOF subsets scoring at or below the test SROCC: {pct:.1%}")
        if pct < 0.05:
            print("-> Test is below almost every random subset of the pool. Likely harder or noisier than the pool (the ensemble should favour test, so this is conservative).")
        else:
            print("-> Test SROCC is within the spread of random subsets. Consistent with noise plus selection optimism; no clear evidence the split is harder.")

    # ---- (3) distribution shift between pool and test -----------------------------------------
    pool = df_all[df_all["split"].isin(["train", "val"]) & df_all["mos"].notna()]
    test = df_all[(df_all["split"] == "test") & df_all["mos"].notna()]
    ks = ks_2samp(pool["mos"], test["mos"])
    cp = pd.Series(class_ids(pool["mos"].values, CFG)).value_counts(normalize=True)
    ct = pd.Series(class_ids(test["mos"].values, CFG)).value_counts(normalize=True)
    comp = pd.DataFrame({"MOS class": [f"{c * CFG['CLASS_BIN_WIDTH']:.1f}" for c in sorted(set(cp.index) | set(ct.index))],
                         "pool share": [cp.get(c, 0) for c in sorted(set(cp.index) | set(ct.index))],
                         "test share": [ct.get(c, 0) for c in sorted(set(cp.index) | set(ct.index))]})
    print(f"\n== (3) Distribution shift ==")
    print(f"MOS mean/std  pool {pool['mos'].mean():.3f}/{pool['mos'].std():.3f}   test {test['mos'].mean():.3f}/{test['mos'].std():.3f}   KS p-value {ks.pvalue:.3f}")
    print(comp.round(3).to_string(index=False))
    t1.to_csv(log_dir / "check3_selection_optimism.csv", index=False)
    return t1

if HAS_LABELS and len(ITER_ROWS) > 0:
    check3_table = check3_val_vs_test()
else:
    print("Check 3 needs the main experiment to have run (ITER_ROWS is empty).")


## 21d-3 · Check 4: does class balancing hurt?

Trains the **same fold** (same partition, same seed) under three sampling settings and compares val and test, plus the mean prediction bias per MOS class on test. Each setting is a full train-until-patience run, so this costs about three iterations. Set `RUN_BALANCE_CHECK = False` to skip.


In [ ]:
RUN_BALANCE_CHECK = True
BALANCE_CHECK_FOLD = 1          # which fold to repeat (1-based)
BALANCE_CHECK_SETTINGS = {
    "natural (no balancing)": {"BALANCE_CLASSES": False},
    "balanced, power 0.5":    {"BALANCE_CLASSES": True, "BALANCE_POWER": 0.5},
    "balanced, power 1.0":    {"BALANCE_CLASSES": True, "BALANCE_POWER": 1.0},
}

def run_one_fold(name: str, overrides: dict, seed: int, fold: int) -> Dict:
    pool = df_all[df_all["split"].isin(["train", "val"]) & df_all["mos"].notna()].reset_index(drop=True)
    te = df_all[(df_all["split"] == "test") & df_all["mos"].notna()].reset_index(drop=True).copy()
    te["split"] = "test"
    tr_idx, va_idx = make_folds(pool, CFG["KFOLD"], CFG["SEED"])[fold - 1]
    tr = pool.iloc[tr_idx].copy(); tr["split"] = "train"
    va = pool.iloc[va_idx].copy(); va["split"] = "val"
    tag = "chk4_" + "".join(ch if ch.isalnum() else "_" for ch in name) + f"_s{seed}_f{fold}"
    with cfg_overrides(overrides):
        set_seed(seed)
        dp = deg_ckpt_path(CFG, f"s{seed}_f{fold}")   # same fold, same training images as the main run
        ensure_deg_pretrained(tr, dp)
        fl = build_loaders(pd.concat([tr, va, te], ignore_index=True), CFG)
        net = init_model(dp)
        _, ck = fit_model(net, fl, tag, CFG["NUM_EPOCHS"], CFG["FREEZE_EPOCHS"], criterion)
        net.load_state_dict(torch.load(ck, map_location=DEVICE)["state_dict"])
        vp, vg = predict_scores(fl["val"], net=net)
        tp, tg = predict_scores(fl["test"], net=net)
        cls = class_ids(tg, CFG)
        bias = {f"bias@{c * CFG['CLASS_BIN_WIDTH']:.1f}": float(np.mean(tp[cls == c] - tg[cls == c]))
                for c in sorted(np.unique(cls))}
    vm, tm = compute_metrics(vp, vg), compute_metrics(tp, tg)
    del net
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return {"setting": name, "stopped_epoch": FIT_INFO[tag]["stopped_epoch"], "best_epoch": FIT_INFO[tag]["best_epoch"],
            **{f"val_{k}": vm[k] for k in ("PLCC", "SROCC", "KRCC", "RMSE")},
            **{f"test_{k}": tm[k] for k in ("PLCC", "SROCC", "KRCC", "RMSE")}, **bias}

if HAS_LABELS and RUN_BALANCE_CHECK and CFG["KFOLD"] >= 2:
    rows = [run_one_fold(n, o, CFG["SEEDS"][0], BALANCE_CHECK_FOLD) for n, o in BALANCE_CHECK_SETTINGS.items()]
    chk4 = pd.DataFrame(rows)
    chk4.to_csv(Path(CFG["OUTPUT_DIR"]) / "logs" / "check4_balancing.csv", index=False)
    show = ["setting", "stopped_epoch", "best_epoch"] + [c for c in chk4.columns if c.startswith(("val_", "test_"))]
    print("\n== Check 4: same fold, different sampling ==")
    print(chk4[show].round(4).to_string(index=False))
    print("\nMean test prediction bias per MOS class (prediction - MOS); positive at low MOS and negative at high MOS means compression to the middle:")
    print(chk4[["setting"] + [c for c in chk4.columns if c.startswith("bias@")]].round(3).to_string(index=False))
    print("\nRead RMSE and PLCC first: if 'natural' beats the balanced settings on both val and test, balancing is hurting. "
          "Judge on val and on test together; one fold is noisy, so repeat with BALANCE_CHECK_FOLD = 2, 3 before deciding.")
else:
    print("Check 4 skipped (needs labels, KFOLD >= 2 and RUN_BALANCE_CHECK = True).")


## 21d-4 · Degradation and complexity ablation (one fold each)

Same fold, same seed, same stratified partition as the main run. Settings: full model (taken from the main run's iteration for that fold), patch degradation loss off, FiLM off, both off, and a shallower Transformer. Each extra setting is one full train-until-patience run. Set `RUN_DEG_ABLATION = False` to skip. A single fold's validation set is small, so treat differences below about 0.02 SROCC as noise and repeat on another fold before concluding.


In [ ]:
RUN_DEG_ABLATION = True
DEG_ABLATION_FOLD = 1
DEG_ABLATION_SETTINGS = {
    "B  no patch degradation loss":         {"HAS_DEGRADATION_GT": False},
    "C  no FiLM (keep patch deg. loss)":    {"USE_FILM": False},
    "D  no degradation mechanisms":         {"HAS_DEGRADATION_GT": False, "USE_FILM": False},
    "E  Transformer depth 2":               {"TRANSFORMER_DEPTH": 2},
}

if HAS_LABELS and RUN_DEG_ABLATION and CFG["KFOLD"] >= 2:
    seed0 = CFG["SEEDS"][0]
    rows = []
    base = [r for r in ITER_ROWS if r["iteration"] == DEG_ABLATION_FOLD and r["seed"] == seed0]
    if base:
        b = base[0]
        rows.append({"setting": "A  full model (main run)", "stopped_epoch": b["stopped_epoch"], "best_epoch": b["best_epoch"],
                     **{k: b[k] for k in b if k.startswith(("val_", "test_"))}})
    else:
        rows.append(run_one_fold("A  full model", {}, seed0, DEG_ABLATION_FOLD))
    for name, ov in DEG_ABLATION_SETTINGS.items():
        rows.append(run_one_fold(name, ov, seed0, DEG_ABLATION_FOLD))
    abl = pd.DataFrame(rows)
    abl.to_csv(Path(CFG["OUTPUT_DIR"]) / "logs" / "check5_degradation_ablation.csv", index=False)
    cols = ["setting", "stopped_epoch", "best_epoch"] + [c for c in abl.columns if c.startswith(("val_", "test_"))]
    print("\n== Degradation / complexity ablation (same fold) ==")
    print(abl[cols].round(4).to_string(index=False))
    print("\nIf B, C or D match or beat A on val AND test, the degradation mechanisms are not earning their complexity. "
          "If A is clearly better on both, keep them. One fold is noisy: repeat with DEG_ABLATION_FOLD = 2, 3.")
else:
    print("Degradation ablation skipped (needs labels, KFOLD >= 2 and RUN_DEG_ABLATION = True).")


## 21d-5 · KADID check: ImageNet only vs ImageNet then KADID

Needs KADID-10k on disk: set `CFG["KADID_DIR"]` to a folder with `images/` and `dmos.csv`, run the notebook once so the KADID pretraining cell (22b) caches `checkpoints/kadid_<backbone>_<size>.pt`, then run this cell. It trains the **same fold** twice: ImageNet-pretrained ResNet50 only, and ImageNet then KADID quality pretraining, both followed by the same fine-tuning on your underwater images. Set `RUN_KADID_CHECK = False` to skip.


In [ ]:
RUN_KADID_CHECK = True
KADID_CHECK_FOLD = 1

if HAS_LABELS and RUN_KADID_CHECK and CFG["KFOLD"] >= 2 and kadid_ckpt_path(CFG).exists():
    seed0 = CFG["SEEDS"][0]
    kad = pd.DataFrame([
        run_one_fold("A  ImageNet only",        {"USE_KADID_INIT": False}, seed0, KADID_CHECK_FOLD),
        run_one_fold("C  ImageNet -> KADID",    {"USE_KADID_INIT": True},  seed0, KADID_CHECK_FOLD),
    ])
    kad.to_csv(Path(CFG["OUTPUT_DIR"]) / "logs" / "check6_kadid.csv", index=False)
    cols = ["setting", "stopped_epoch", "best_epoch"] + [c for c in kad.columns if c.startswith(("val_", "test_"))]
    print("\n== KADID check (same fold, same seed) ==")
    print(kad[cols].round(4).to_string(index=False))
    print("\nOne fold is noisy: treat differences below about 0.02 SROCC as noise and repeat with KADID_CHECK_FOLD = 2, 3. "
          "KADID distortions are synthetic, so a gain is not guaranteed on real underwater degradation.")
else:
    print("KADID check skipped. Needs labels, KFOLD >= 2, RUN_KADID_CHECK = True and a cached KADID checkpoint "
          "(set CFG['KADID_DIR'] and run section 22b first).")


## 21d-6 · Backbone check: ResNet50 patches vs pretrained Swin

Trains the **same fold** (same partition and seed) with the ResNet50 patch encoder and with ImageNet-pretrained Swin-T and Swin-S used as whole-image encoders (token grids pooled to the same 84-token layout). Needs the timm Swin weights to download on first use. Set `RUN_BACKBONE_CHECK = True` to run it; each setting is a full train-until-patience run.


In [ ]:
RUN_BACKBONE_CHECK = False
BACKBONE_CHECK_FOLD = 1
BACKBONE_CHECK_SETTINGS = {
    "A  ResNet50 patches (current)":   {"CNN_BACKBONE": "resnet50"},
    "B  Swin-T, whole-image grid":     {"CNN_BACKBONE": "swin_t", "CNN_INPUT_SIZE": 224},
    "C  Swin-S, whole-image grid":     {"CNN_BACKBONE": "swin_s", "CNN_INPUT_SIZE": 224},
}

if HAS_LABELS and RUN_BACKBONE_CHECK and CFG["KFOLD"] >= 2:
    seed0 = CFG["SEEDS"][0]
    bb = pd.DataFrame([run_one_fold(n, o, seed0, BACKBONE_CHECK_FOLD) for n, o in BACKBONE_CHECK_SETTINGS.items()])
    bb.to_csv(Path(CFG["OUTPUT_DIR"]) / "logs" / "check7_backbones.csv", index=False)
    cols = ["setting", "stopped_epoch", "best_epoch"] + [c for c in bb.columns if c.startswith(("val_", "test_"))]
    print("\n== Backbone check (same fold, same seed) ==")
    print(bb[cols].round(4).to_string(index=False))
    print("\nOne fold is noisy: treat differences below about 0.02 SROCC as noise and repeat with BACKBONE_CHECK_FOLD = 2, 3.")
else:
    print("Backbone check off. Set RUN_BACKBONE_CHECK = True to run it.")


## 21e · Label audit: worst-error images (v3)

Out-of-fold errors averaged over seeds. The named images are checked here. A high error that repeats across seeds is either a label problem or a genuinely hard image. Look at the thumbnail before changing any label.


In [ ]:
NAMED = ["0064.png", "0688.png", "0011.png", "0855.png"]


def label_audit(results: List[Dict], top_n: int = 10) -> pd.DataFrame:
    '''
    Per-image out-of-fold error, averaged over seeds. A large error that repeats across
    seeds and folds points to a label problem or a genuinely hard image.
    '''
    frames = [pd.DataFrame({"image_name": r["oof_names"], "mos": r["oof_gt"],
                            "pred_cal": r["oof_cal_pred"], "seed": r["seed"]}) for r in results]
    a = pd.concat(frames, ignore_index=True)
    a["abs_err"] = (a["pred_cal"] - a["mos"]).abs()
    agg = (a.groupby("image_name")
             .agg(mos=("mos", "first"), pred_mean=("pred_cal", "mean"),
                  abs_err_mean=("abs_err", "mean"), abs_err_std=("abs_err", "std"),
                  n_seeds=("seed", "nunique"))
             .sort_values("abs_err_mean", ascending=False))
    print(f"Top {top_n} worst OOF images (mean |error| over seeds):")
    print(agg.head(top_n).round(4).to_string())
    print("\nNamed images (NaN = not in the train+val pool, e.g. it is in test):")
    print(agg.reindex(NAMED).round(4).to_string())
    return agg


def show_thumbs(agg: pd.DataFrame, names) -> None:
    names = [n for n in names if n in agg.index]
    if not names:
        print("None of the named images are in the OOF pool.")
        return
    paths = dict(zip(df_all["image_name"], df_all["image_path"]))
    fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 4.6))
    axes = np.atleast_1d(axes)
    for ax, n in zip(axes, names):
        ax.imshow(Image.open(paths[n]).convert("RGB")); ax.axis("off")
        ax.set_title(f"{n}\nMOS={agg.loc[n, 'mos']:.3f}  pred={agg.loc[n, 'pred_mean']:.3f}", fontsize=9)
    plt.tight_layout()
    plt.savefig(Path(CFG["OUTPUT_DIR"]) / "label_audit.png", dpi=110)
    plt.show()


if RESULTS:
    AUDIT = label_audit(RESULTS)
    show_thumbs(AUDIT, NAMED)
    AUDIT.to_csv(Path(CFG["OUTPUT_DIR"]) / "oof_label_audit.csv")
else:
    print("Label audit needs RESULTS from the main experiment.")


## 21e-2 · Test-set error audit (inspection only)

Table of the hardest test images: filename, MOS, raw prediction, calibrated prediction and absolute error, plus a thumbnail grid and the mean error per MOS class. Use it to look for label inconsistencies, unusual scenes or acquisition differences. Do **not** remove or relabel test images based on this and then re-report the test score; that would turn the test set into a tuning set.


In [ ]:
def test_audit(results: List[Dict], top_n: int = 15, show: int = 12) -> pd.DataFrame:
    r0 = results[0]
    d = pd.DataFrame({"image_name": r0["test_names"], "mos": r0["test_gt"],
                      "pred_raw": r0["test_raw_pred"], "pred_cal": r0["test_cal_pred"]})
    d["abs_err_raw"] = (d["pred_raw"] - d["mos"]).abs()
    d["abs_err_cal"] = (d["pred_cal"] - d["mos"]).abs()
    d = d.sort_values("abs_err_cal", ascending=False).reset_index(drop=True)
    d.to_csv(Path(CFG["OUTPUT_DIR"]) / "test_audit.csv", index=False)
    print(f"Top {top_n} test images by calibrated absolute error:")
    print(d.head(top_n).round(3).to_string(index=False))

    cls = class_ids(d["mos"].values, CFG)
    by = (d.assign(mos_class=[f"{c * CFG['CLASS_BIN_WIDTH']:.1f}" for c in cls],
                   bias_raw=d["pred_raw"] - d["mos"])
            .groupby("mos_class").agg(n=("mos", "size"), mean_bias_raw=("bias_raw", "mean"),
                                      mean_abs_err_raw=("abs_err_raw", "mean")))
    print("\nTest error by MOS class (bias = prediction - MOS):"); print(by.round(3).to_string())

    paths = dict(zip(df_all["image_name"], df_all["image_path"]))
    top = d.head(show)
    cols = 4; rows_ = int(np.ceil(len(top) / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(4 * cols, 4.2 * rows_))
    for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
    for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), top.iterrows()):
        ax.imshow(Image.open(paths[r["image_name"]]).convert("RGB"))
        ax.set_title(f"{r['image_name']}\nMOS={r['mos']:.2f} raw={r['pred_raw']:.2f} cal={r['pred_cal']:.2f}", fontsize=8)
    plt.tight_layout()
    plt.savefig(Path(CFG["OUTPUT_DIR"]) / "test_audit.png", dpi=110); plt.show()
    return d

if RESULTS:
    TEST_AUDIT = test_audit(RESULTS)
else:
    print("Test audit needs RESULTS from the main experiment.")


## 21f · Ablations (off by default)

Set `RUN_ABLATIONS = True` to run each configuration as seeds x k-fold. A through F cover: Huber vs MSE, FiLM on/off, resolution 384, DINOv2 full fine-tune, and DINOv2 frozen with a light head. Each one costs as much as the main experiment.


In [ ]:
# Each ablation is a full seeds x k-fold run, so this is expensive. Off by default.
ABLATIONS = {
    "A  baseline: ResNet50 @256, Huber, FiLM":  {},
    "B  MSE loss (v2 loss)":                    {"MOS_LOSS": "mse"},
    "C  no FiLM (no degradation feedback)":     {"USE_FILM": False},
    "D  IMAGE_SIZE 384":                        {"IMAGE_SIZE": 384},
    "E  DINOv2-S/14, full fine-tune @224":      {"CNN_BACKBONE": "dinov2_vits14", "CNN_INPUT_SIZE": 224},
    "F  DINOv2-S/14, frozen + light head":      {"CNN_BACKBONE": "dinov2_vits14", "CNN_INPUT_SIZE": 224,
                                                 "FREEZE_EPOCHS": 10**9},
    "G  Swin-T (pretrained), whole-image grid": {"CNN_BACKBONE": "swin_t", "CNN_INPUT_SIZE": 224},
    "H  Swin-S (pretrained), whole-image grid": {"CNN_BACKBONE": "swin_s", "CNN_INPUT_SIZE": 224},
}
RUN_ABLATIONS = False

ABL_ROWS = []
if RUN_ABLATIONS and HAS_LABELS:
    for name, ov in ABLATIONS.items():
        print(f"\n################  ABLATION {name}  ################")
        with cfg_overrides(ov):
            res = run_all_seeds(CFG["SEEDS"])
            t_cal = summarise_runs(res, "test_cal")
        row = {"config": name}
        for _, rr in t_cal.iterrows():
            row[f"{rr['metric']} (cal)"] = rr["mean±std"]
        ABL_ROWS.append(row)
    ABL_TABLE = pd.DataFrame(ABL_ROWS)
    print("\n=== Ablation summary (TEST, calibrated, mean ± std over seeds) ===")
    print(ABL_TABLE.to_string(index=False))
    ABL_TABLE.to_csv(Path(CFG["OUTPUT_DIR"]) / "ablation_summary.csv", index=False)
else:
    print("Ablations off. Set RUN_ABLATIONS = True to run them (each is seeds x k-fold).")


## 25 . Explainability Maps

The model naturally produces spatial attribution maps for each input image.

For each patch:
$$c_i = \alpha_i q_i \quad \text{(quality contribution)}$$
$$r_i = \alpha_i (1-q_i) \quad \text{(degradation contribution)}$$

These are **model attributions**, not physical ground-truth degradation measurements. They indicate which regions the model considers important and which regions it predicts are degraded.

**Visualization outputs for each image:**
1. Original image  
2. Patch quality map ($q_i$)  
3. Saliency/importance map ($\alpha_i$)  
4. Quality contribution map ($c_i = \alpha_i q_i$)  
5. Degradation contribution map ($r_i = \alpha_i(1-q_i)$)  
6. Predicted global quality score $\hat{Q}$


In [ ]:
def reconstruct_map(
    values   : np.ndarray,    # (N_total,) per-patch values
    slices   : dict,          # {patch_size: (start, end)}
    patch_sizes: list,
    img_size : int,
) -> Dict[int, np.ndarray]:
    '''
    Reconstruct per-patch values back into 2D spatial maps,
    one map per scale.

    Returns: {patch_size: 2D array of shape (img_size, img_size)}
    '''
    maps = {}
    for p in patch_sizes:
        start, end = slices[p]
        vals = values[start:end]          # (N_s,)
        n = img_size // p
        grid = vals.reshape(n, n)
        # Upsample to full image size using nearest-neighbour
        upsampled = np.kron(grid, np.ones((p, p)))
        maps[p] = upsampled
    return maps


def visualize_explainability(
    model      : nn.Module,
    image_path : str,
    cfg        : dict,
    save_path  : Optional[str] = None,
) -> None:
    '''
    Generate and display all explainability maps for a single image.
    '''
    model.eval()

    # Load and preprocess
    tfm = get_transforms(cfg["IMAGE_SIZE"], train=False)
    img_pil = Image.open(image_path).convert("RGB").resize(
        (cfg["IMAGE_SIZE"], cfg["IMAGE_SIZE"])
    )
    img_tensor = tfm(img_pil).unsqueeze(0).to(DEVICE)  # (1, 3, H, W)

    with torch.no_grad():
        out = model(img_tensor)

    Q_hat  = out["Q_hat"][0].item()
    q      = out["q"][0, :, 0].cpu().numpy()        # (N_total,)
    alphas = out["alphas"][0, :, 0].cpu().numpy()   # (N_total,)

    quality_contrib = alphas * q                    # c_i
    degradation_contrib = alphas * (1.0 - q)       # r_i

    slices_cpu = {k: v for k, v in out["slices"].items()}
    H = W = cfg["IMAGE_SIZE"]
    patch_sizes = cfg["PATCH_SIZES"]

    # Reconstruct maps for the finest scale (most informative visually)
    p_vis = patch_sizes[0]  # 32x32 gives 64 patches = highest resolution
    start, end = slices_cpu[p_vis]

    def to_map(arr):
        n = H // p_vis
        return np.kron(arr[start:end].reshape(n,n), np.ones((p_vis,p_vis)))

    q_map   = to_map(q)
    a_map   = to_map(alphas)
    c_map   = to_map(quality_contrib)
    d_map   = to_map(degradation_contrib)

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    titles = [
        "Original",
        f"Patch Quality (q)\nscale={p_vis}",
        f"Importance (alpha)\nscale={p_vis}",
        f"Quality Contribution\nci=alphaiqi",
        f"Degradation Contribution\nri=alphai(1-qi)",
    ]
    contents = [np.array(img_pil), q_map, a_map, c_map, d_map]
    cmaps    = [None, "RdYlGn", "hot", "RdYlGn", "RdYlBu_r"]

    for ax, title, content, cmap in zip(axes, titles, contents, cmaps):
        if cmap is None:
            ax.imshow(content)
        else:
            im = ax.imshow(content, cmap=cmap, vmin=0, vmax=content.max())
            plt.colorbar(im, ax=ax, fraction=0.046)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.suptitle(
        f"Explainability Maps  |  Predicted Quality: {Q_hat:.3f}\n"
        f"WARNING Maps reflect model attributions, not physical ground-truth degradation.",
        fontsize=10, y=1.02
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved to {save_path}")
    plt.show()


# Run on first sample image
_sample = df_all["image_path"].iloc[0]
visualize_explainability(
    model, _sample, CFG,
    save_path=str(Path(CFG["OUTPUT_DIR"]) / "explainability_sample.png")
)


## 26 . Ablation Study Framework

This section defines the eight ablation models described in the design document Section8 (equations 103-110).

| Model | Description | What it tests |
|-------|-------------|---------------|
| M0 | Global CNN | Baseline - no patch decomposition |
| M1 | Patch CNN + Average Pooling | Uniform patch aggregation |
| M2 | Patch CNN + Attention Pooling | Learned pooling without saliency |
| M3 | Patch CNN + Saliency Pooling | Saliency-based importance |
| M4 | Saliency + Degradation Pooling | Degradation-aware importance |
| M5 | Multi-Scale Patch Model | Multi-scale without Transformer |
| M6 | Multi-Scale + Transformer | Context modeling without degradation |
| M7 | Full Hybrid Model | Complete proposed architecture |

**Protocol:** Same data split, same seed, same training hyperparameters, same loss function across all ablation models.

**Shared add-ons:** M1, M2, M6 and M7 use the same squeeze-and-excitation block (`CFG["USE_SE"]`); M6 and M7 both use per-image scale attention. M0 is a plain global-CNN baseline without them.

> **Research integrity note:** The classes below are ready to train; reported numbers depend on actual experimental runs and must not be fabricated.


In [ ]:
# -- M0: Global CNN baseline -------------------------------------------------
class GlobalCNNModel(nn.Module):
    '''M0: Resize image, CNN features, global average pool, MLP quality head.'''
    def __init__(self, cfg: dict):
        super().__init__()
        backbone_name = cfg["CNN_BACKBONE"]
        weights = "IMAGENET1K_V1" if cfg["CNN_PRETRAINED"] else None
        if backbone_name == "resnet18":
            base = tvm.resnet18(weights=weights)
            cnn_out = 512
        else:
            base = tvm.resnet50(weights=weights)
            cnn_out = 2048
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(cnn_out, 256), nn.GELU(), nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, images):
        f = self.features(images)
        return {"Q_hat": self.head(f).squeeze(-1)}


# -- M1: Patch CNN + Average Pooling ----------------------------------------
class PatchCNNAvgPool(nn.Module):
    '''M1: Fixed 32x32 patches, CNN features, uniform average pooling.'''
    def __init__(self, cfg: dict):
        super().__init__()
        self.patch_size = 32
        self.cnn = CNNPatchEncoder(cfg["CNN_BACKBONE"], cfg["EMBED_DIM"], cfg["CNN_PRETRAINED"], use_se=cfg.get("USE_SE", True))
        self.quality_head = nn.Sequential(
            nn.Linear(cfg["EMBED_DIM"], 1), nn.Sigmoid()
        )
    def forward(self, images):
        B = images.shape[0]
        patches = extract_patches_multiscale(images, [self.patch_size])
        pts = patches[self.patch_size]
        N = pts.shape[1]
        feats = self.cnn(pts.reshape(B*N, 3, self.patch_size, self.patch_size))
        feats = feats.reshape(B, N, -1)
        q = self.quality_head(feats).squeeze(-1)    # (B, N)
        Q_hat = q.mean(dim=1)                        # uniform average
        return {"Q_hat": Q_hat}


# -- M2: Patch CNN + Attention Pooling --------------------------------------
class PatchCNNAttnPool(nn.Module):
    '''M2: Learned attention weights, no explicit saliency branch.'''
    def __init__(self, cfg: dict):
        super().__init__()
        self.patch_size = 32
        d = cfg["EMBED_DIM"]
        self.cnn = CNNPatchEncoder(cfg["CNN_BACKBONE"], d, cfg["CNN_PRETRAINED"], use_se=cfg.get("USE_SE", True))
        self.quality_head = nn.Sequential(nn.Linear(d, 1), nn.Sigmoid())
        self.attn_head    = nn.Linear(d, 1)
    def forward(self, images):
        B = images.shape[0]
        patches = extract_patches_multiscale(images, [self.patch_size])
        pts = patches[self.patch_size]
        N = pts.shape[1]
        feats = self.cnn(pts.reshape(B*N, 3, self.patch_size, self.patch_size)).reshape(B,N,-1)
        q     = self.quality_head(feats).squeeze(-1)
        alpha = F.softmax(self.attn_head(feats).squeeze(-1), dim=1)
        Q_hat = (alpha * q).sum(dim=1)
        return {"Q_hat": Q_hat}


# -- M6: Multi-Scale + Transformer (no degradation branch) ------------------
class MultiScaleTransformerNoDeg(nn.Module):
    '''M6: Full transformer but quality+saliency only (no degradation branch).'''
    def __init__(self, cfg: dict):
        super().__init__()
        d = cfg["EMBED_DIM"]
        self.patch_sizes = cfg["PATCH_SIZES"]
        self.cnn = CNNPatchEncoder(cfg["CNN_BACKBONE"], d, cfg["CNN_PRETRAINED"], use_se=cfg.get("USE_SE", True))
        self.transformer = PatchTransformerEncoder(
            cfg["PATCH_SIZES"], cfg["IMAGE_SIZE"], d,
            cfg["TRANSFORMER_DEPTH"], cfg["TRANSFORMER_HEADS"],
            cfg["TRANSFORMER_FF_DIM"], cfg["TRANSFORMER_DROPOUT"]
        )
        self.q_head = nn.Sequential(nn.Linear(d, 1), nn.Sigmoid())
        self.a_head = nn.Linear(d, 1)
        self.scale_logits = nn.Parameter(torch.zeros(len(cfg["PATCH_SIZES"])))
        # Per-image scale attention (zero-init), same as the full hybrid model
        self.scale_attn = nn.Linear(d, 1)
        nn.init.zeros_(self.scale_attn.weight)
        nn.init.zeros_(self.scale_attn.bias)

    def forward(self, images):
        B = images.shape[0]
        patches = extract_patches_multiscale(images, self.patch_sizes)
        scale_feats = {}
        for p in self.patch_sizes:
            pts = patches[p]; N_s = pts.shape[1]
            f = self.cnn(pts.reshape(B*N_s, 3, p, p)).reshape(B, N_s, -1)
            scale_feats[p] = f
        h, slices = self.transformer(scale_feats)
        q = self.q_head(h).squeeze(-1)
        Qs, ctx = [], []
        for i, p in enumerate(self.patch_sizes):
            st, en = slices[p]
            h_s = h[:, st:en, :]
            q_s = q[:, st:en]
            alpha_s = F.softmax(self.a_head(h_s).squeeze(-1), dim=1)
            Qs.append((alpha_s * q_s).sum(1))
            ctx.append(self.scale_attn(h_s.mean(dim=1)))                       # (B, 1)
        lambdas_b = F.softmax(self.scale_logits.unsqueeze(0) + torch.cat(ctx, dim=-1), dim=-1)   # (B, S)
        Q_hat = (torch.stack(Qs, dim=-1) * lambdas_b).sum(-1)
        return {"Q_hat": Q_hat}


# -- Ablation runner ---------------------------------------------------------
ABLATION_MODELS = {
    "M0_GlobalCNN"          : GlobalCNNModel,
    "M1_PatchAvgPool"       : PatchCNNAvgPool,
    "M2_PatchAttnPool"      : PatchCNNAttnPool,
    "M6_MultiScaleTransformer": MultiScaleTransformerNoDeg,
    "M7_FullHybrid"         : HybridUIQAModel,   # already defined
}

def run_ablation(
    ablation_models : dict,
    cfg             : dict,
    loaders         : dict,
    n_epochs        : int = 5,    # Use fewer epochs for ablation sweeps
    run_training    : bool = False,
) -> pd.DataFrame:
    '''
    Train and evaluate each ablation model.

    Args:
        run_training: Set True to actually train. False = architecture-only demo.
                      Training requires MOS labels.
    '''
    results = []
    for name, ModelClass in ablation_models.items():
        print(f"\n-- {name} -----------------------------")
        m = ModelClass(cfg).to(DEVICE)
        n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
        print(f"   Parameters: {n_p:,}")

        if run_training and HAS_LABELS and "train" in loaders and "val" in loaders:
            opt = torch.optim.AdamW(m.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
            sc  = GradScaler(enabled=cfg["USE_AMP"] and DEVICE.type=="cuda")
            for ep in range(n_epochs):
                train_epoch(m, loaders["train"], opt, criterion, sc, cfg)
            _, preds, gts = validate_epoch(m, loaders["val"], criterion, cfg)
            metrics = compute_metrics(preds, gts)
        else:
            metrics = {"PLCC": float("nan"), "SROCC": float("nan"),
                       "RMSE": float("nan"), "KRCC": float("nan")}
            if not run_training:
                print("   (training skipped - set run_training=True to train)")

        row = {"Model": name, "Params": n_p, **metrics}
        results.append(row)
        print(f"   {metrics}")

    return pd.DataFrame(results)


# Run ablation in architecture-demo mode (no training).
# Set run_training=True when MOS labels are available and you want full ablation results.
ablation_df = run_ablation(ABLATION_MODELS, CFG, loaders, n_epochs=2, run_training=False)
print("\n-- Ablation Summary ----------------------------------------------")
print(ablation_df.to_string(index=False))
print("\nWARNING  Metrics show NaN because training was skipped.")
print("   Set run_training=True with a labelled dataset for real ablation results.")
print("   Do NOT interpret NaN as 'model failed'; it means training was not run.")


## 27 . Save Predictions

In [ ]:
def save_predictions(
    model     : nn.Module,
    loader    : DataLoader,
    df_split  : pd.DataFrame,
    save_path : str,
) -> pd.DataFrame:
    model.eval()
    all_preds = []

    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc="Inference"):
            imgs = imgs.to(DEVICE)
            if CFG.get("USE_TTA", False):
                q_hat = tta_batch_predict(model, imgs, CFG)
            else:
                q_hat = model(imgs)["Q_hat"]
            all_preds.extend(q_hat.float().cpu().numpy().tolist())

    df_out = df_split.reset_index(drop=True).copy()
    df_out["predicted_quality"] = all_preds
    df_out.to_csv(save_path, index=False)
    print(f"Predictions saved to {save_path}")
    return df_out


if "test" in loaders and CFG["KFOLD"] < 2:
    df_test = df_all[df_all["split"]=="test"].reset_index(drop=True)
    pred_df = save_predictions(
        model, loaders["test"], df_test,
        str(Path(CFG["OUTPUT_DIR"]) / "test_predictions.csv")
    )
    print(pred_df[["image_name","mos","predicted_quality"]].head(10).to_string(index=False))


## 28 . Reproducibility Checklist

| Item | Status |
|------|--------|
| Random seed fixed before data operations | v `SEED=42` |
| Seed documented | v Stored in checkpoint |
| Dataset split at image level before patch extraction | v |
| No patches from same image in multiple splits | v Verified |
| ImageNet normalization applied consistently | v |
| CNN pretrained weights source documented | v ImageNet1K |
| Checkpoint saves model config alongside weights | v |
| All hyperparameters in single `CFG` dict | v |
| Loss weights configurable | v |
| Degradation loss auto-disabled without GT | v |
| Ablation uses same split and protocol | v |
| No labels fabricated | v |
| Mixed precision optional (not affecting results determinism) | v |


## 29 . Research Limitations and Interpretation

### Limitations of this implementation

1. **No patch-level MOS labels.** The model learns latent patch quality through image-level weak supervision (MIL). Patch-quality maps are model attributions, not verified ground truth.

2. **Degradation branch is self-supervised.** Without per-patch degradation annotations, the degradation branch ($\mathbf{r}_i$) learns representations correlated with quality rather than physically calibrated degradation estimates. The design document Section3.9 explicitly allows for this via pseudo-labels or self-supervision.

3. **ImageNet domain gap.** The CNN backbone is pretrained on terrestrial ImageNet images. Underwater images have a different colour distribution. Fine-tuning on underwater data is recommended.

4. **Patch boundary effects.** Non-overlapping patches may split semantic objects at boundaries. Overlapping patches (as mentioned in the design document Section2.2) would mitigate this at higher computational cost.

5. **Scale weight interpretability.** The scale weights $\lambda_s$ are image-adaptive: a softmax over a learnable global bias plus a per-image term computed from the mean token of each scale. The logged $\lambda$ is the batch mean, so it summarises, but does not describe, any single image.

6. **Experimental results not reported.** This is a prototype. PLCC, SROCC, KRCC, and RMSE values depend on the actual underwater image dataset and MOS collection methodology used.

### Interpreting the maps

The patch-quality map $q_i$, saliency map $\alpha_i$, and degradation contribution map $r_i = \alpha_i(1-q_i)$ are **model predictions/attributions**:
- They reflect what the model has learned to associate with quality scores.
- They do NOT constitute physical measurements of light absorption, scattering, or turbidity.
- They should be validated against human psychophysical experiments for research use.

### What this prototype demonstrates

- v That the three-branch + Transformer hybrid is architecturally sound and implementable.
- v That patch extraction, CNN encoding, Transformer context modeling, and importance-weighted pooling compose coherently.
- v That the model produces explainable spatial outputs alongside its quality score.
- x Whether this architecture achieves state-of-the-art performance - **this requires experimental validation on a benchmark dataset with human MOS labels**.
